In [ ]:

import os, sys
import numpy as np
import muon as mu
import pandas as pd
import torch
import importlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import gtfparse
from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader, Subset

DATA_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data")
PROJECT_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/dev/notebooks/simple_model_testing")
sys.path.append(str(PROJECT_DIR))

import utils

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):


# Testing a simple classifier

I am going to see if I can simplify my approach for GRN inference now that I'm playing around with using a classifier.

Let's go back to the basics to figure out what information I have, what information I need, and how I can approach this problem.

---

## Information I have available:

### Gene Identity Information
I know which genes are present in the dataset and how strongly they are being expressed. 

I can use this to let the model learn how each TF and TG uniquely affects the regulatory landscape.

**Variables:**
- TF Identity
- TF Expression
- TF Protein Structure
- TG Identity
- TG Expression

### ATAC Information
I know which regions of the genome are accessible from the ATAC data.

The model can use this information to determine which TFs are likely to bind to an open chromatin region near a gene TSS. 

**Variables:**
- ATAC Peak Location
- ATAC peak DNA sequence

### Dataset Information
I know the organism , cell type, and experimental conditions that the samples came from.

I can use this to inform the model on how to update it's predictions based on the context.

**Variables:**
- Cell Type
- Organism
- Experimental condition

### Other Information Available:
- Prior knowledge information from KEGG, Reactome, and STRING
- ChIP-seq TF-DNA binding information
- Hi-C DNA-DNA interaction maps

## How can I combine these variables to get useful biological information?

1. Peak to TG TSS distance
2. TF-DNA interaction based on sequence and protein structure
3. Relationship between TF expression, peak accessibility, and TG expression

## Gathering information:

### 1. RNA and ATAC Expression Data

I can use the processed data for E7.5_rep1 as the starting point.

In [4]:
adata = mu.read_h5mu(PROJECT_DIR / "data" / "E7.5_rep1.h5mu")
adata

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


MuData object with n_obs × n_vars = 8617 × 202338
  var:	'gene_ids', 'feature_types', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
  2 modalities
    rna:	5678 x 2453
      obs:	'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt'
      var:	'gene_ids', 'feature_types', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
      uns:	'hvg', 'log1p'
      layers:	'counts'
    atac:	8138 x 199885
      obs:	'n_genes_by_counts', 'total_counts', 'n_counts', 'leiden'
      var:	'gene_ids', 'feature_types', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
      uns:	'atac', 'hvg', 'leiden', 'leiden_colors', 'log1p', 'lsi', 'neighbors', 'pca', 'umap'
      obsm:	'X_lsi', 'X_pca', 'X_umap'
      varm:	'LSI', 'PCs'
      layers:	'counts'
      obsp:	'connectivities', 'distances'

I also have the RNA and ATAC pseudobulk datasets

In [6]:
atac_pseudobulk = pd.read_parquet(PROJECT_DIR / "data" / "ATAC_data" / "RE_pseudobulk.parquet")
display(atac_pseudobulk.head())
print(f"Peaks: {atac_pseudobulk.shape[0]}, Cells: {atac_pseudobulk.shape[1]}")

,AAACAGCCAAACTCAT-1,AAACATGCAACCTGGT-1,AAACATGCAATGAATG-1,AAACATGCAATTATGC-1,AAACATGCACCTATAG-1,AAACATGCAGGATAAC-1,AAACCAACATAATCGT-1,AAACCGAAGAGGGACT-1,AAACCGCGTGAACAAA-1,AAACCGCGTTAACGAT-1,...,TTTGTGGCAATAATGG-1,TTTGTGGCACAATGCC-1,TTTGTGGCAGCACCAT-1,TTTGTGGCAGCACGTT-1,TTTGTGGCATGAATAG-1,TTTGTGTTCTGTGAGT-1,TTTGTTGGTACAATGT-1,TTTGTTGGTCCTTCTC-1,TTTGTTGGTTAAGCCA-1,TTTGTTGGTTAATGAC-1
chr1:3035460-3036350,-0.149983,-0.149983,-0.138368,-0.149073,0.616351,-0.149983,-0.149983,-0.149983,-0.149983,0.408187,...,-0.098493,-0.149983,0.089859,-0.029984,-0.121615,0.239383,-0.149983,-0.091435,-0.149983,-0.149983
chr1:3044677-3045614,-0.133086,0.164751,-0.129855,-0.133086,-0.133086,-0.001326,0.193204,-0.133086,-0.133086,-0.133086,...,1.098804,-0.133086,-0.133086,-0.133086,-0.090196,-0.133086,0.033749,-0.113548,1.135969,0.119937
chr1:3062482-3063387,-0.177036,-0.177036,-0.177036,0.055561,-0.121887,-0.177036,0.358991,-0.177036,-0.151576,0.226338,...,-0.146005,-0.177036,-0.151802,-0.167863,-0.175739,-0.048367,-0.177036,0.017758,-0.173306,-0.102419
chr1:3072145-3073018,-0.133391,1.011918,-0.094196,-0.051031,-0.099223,-0.133391,-0.059047,-0.133391,-0.133391,-0.133391,...,0.340874,-0.130360,-0.133391,-0.133391,-0.133391,-0.133391,-0.133391,-0.126633,1.583125,-0.133391
chr1:3191389-3192286,-0.180791,-0.206165,-0.079078,0.671649,0.046988,-0.206165,-0.099598,-0.206165,-0.041705,-0.148497,...,-0.073659,-0.203599,-0.130258,0.116267,0.045015,-0.145928,-0.192585,0.684141,-0.182941,-0.175843


Peaks: 199885, Cells: 5199


In [7]:
rna_pseudobulk = pd.read_parquet(PROJECT_DIR / "data" / "RNA_data" / "TG_pseudobulk.parquet")
display(rna_pseudobulk.head())
print(f"Genes: {rna_pseudobulk.shape[0]}, Cells: {rna_pseudobulk.shape[1]}")

,AAACAGCCAAACTCAT-1,AAACATGCAACCTGGT-1,AAACATGCAATGAATG-1,AAACATGCAATTATGC-1,AAACATGCACCTATAG-1,AAACATGCAGGATAAC-1,AAACCAACATAATCGT-1,AAACCGAAGAGGGACT-1,AAACCGCGTGAACAAA-1,AAACCGCGTTAACGAT-1,...,TTTGTGGCAATAATGG-1,TTTGTGGCACAATGCC-1,TTTGTGGCAGCACCAT-1,TTTGTGGCAGCACGTT-1,TTTGTGGCATGAATAG-1,TTTGTGTTCTGTGAGT-1,TTTGTTGGTACAATGT-1,TTTGTTGGTCCTTCTC-1,TTTGTTGGTTAAGCCA-1,TTTGTTGGTTAATGAC-1
original_var_name,,,,,,,,,,,,,,,,,,,,,
0610010F05RIK,-0.293736,-0.482003,0.035822,-0.257651,-0.194493,0.343746,-0.087335,-0.164938,-0.088705,0.350599,...,-0.100859,0.319101,0.091830,-0.232626,0.025435,0.539317,0.128127,-0.265299,0.226345,-0.092356
0610040J01RIK,-0.158522,-0.400258,0.046670,0.025062,-0.295229,-0.411964,0.695844,0.575298,-0.071259,0.121791,...,0.976641,1.099468,-0.103735,0.003339,-0.369843,-0.290606,-0.383585,-0.361580,1.548686,-0.328809
1010001N08RIK,-0.191342,2.651695,0.350153,-0.155214,-0.182251,0.069193,-0.019593,-0.194774,-0.094297,-0.194774,...,0.204189,0.229093,-0.194774,0.057130,0.067004,-0.194774,-0.161628,-0.112418,-0.004261,-0.138205
1190005I06RIK,-0.041903,-0.205857,-0.098478,-0.131061,-0.034418,-0.195359,-0.179338,-0.027725,-0.055397,-0.081109,...,-0.185821,0.192008,0.181546,-0.190065,-0.179925,-0.010806,-0.164151,-0.195406,0.084243,-0.108067
1600010M07RIK,0.154519,-0.695117,0.292645,-0.332464,-0.145477,0.815248,-0.304740,0.309819,-0.089212,0.416993,...,-0.248529,-0.091530,0.061750,-0.033387,-0.104972,-0.084400,0.055898,0.152006,-0.087099,-0.347106


Genes: 2453, Cells: 5199


### 2. Peak to Gene TSS Distance

I also know the distance between each peak in the ATAC dataset to each gene's TSS within 1Mb

In [8]:
peak_to_gene_distance = pd.read_parquet(PROJECT_DIR / "data" / "ATAC_data" / "peak_to_gene_dist.parquet")
peak_to_gene_distance = peak_to_gene_distance[["peak_id", "target_id", "TSS_dist"]]
display(peak_to_gene_distance.head())
print(f"Peaks: {peak_to_gene_distance['peak_id'].nunique()}, Genes: {peak_to_gene_distance['target_id'].nunique()}, Total pairs: {len(peak_to_gene_distance)}")

,peak_id,target_id,TSS_dist
0,chr9:88480998-88481720,9430037G07RIK,0
1,chr9:50746017-50746871,ALG9,0
2,chr3:89868325-89869122,ATP8B2,0
3,chr6:113283221-113284098,BRPF1,0
4,chr9:107928096-107928874,GM20529,0


Peaks: 196885, Genes: 48772, Total pairs: 196885


### 3. Peak DNA sequence information

I can get the DNA sequence information for each peak and create a one-hot encoding of the sequence.

In [9]:
genome_fasta_path = DATA_DIR / "genome_data" / "reference_genome" / "mm10" / "mm10.fa"
chrom_sizes_path = DATA_DIR / "genome_data" / "reference_genome" / "mm10" / "mm10.chrom.sizes"

In [7]:
first_peak = atac_pseudobulk.index[0]

first_peak_seq = utils.load_peak_sequence(genome_fasta_path, first_peak)
print(f"First peak: {first_peak}")
print(f"First peak sequence: {first_peak_seq[:10]}...{first_peak_seq[-10:]} (length: {len(first_peak_seq)})")

first_peak_onehot = utils.onehot_dna_sequence(first_peak_seq)
display(first_peak_onehot[:5])  # Show first 5 rows of the one-hot encoded matrix
print(f"First peak one-hot shape: {first_peak_onehot.shape}")

First peak: chr1:3035460-3036350
First peak sequence: TATTTTATTT...GTTTTTAATA (length: 890)


array([[0., 0., 0., 1.],
       [1., 0., 0., 0.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.]], dtype=float32)

First peak one-hot shape: (890, 4)


### 4. TF Protein Sequence Information

In [10]:
tf_name_file = DATA_DIR / "databases" / "motif_information" / "mm10" / "TF_Information_all_motifs.txt"
gene_ref_file = DATA_DIR / "genome_data" / "genome_annotation" / "mm10" / "Mus_musculus.GRCm39.115.gtf.gz"

In [11]:
gene_ref_df = gtfparse.read_gtf(
    gene_ref_file,
    result_type="pandas"
    )

display(gene_ref_df.head())

INFO:root:Extracted GTF attributes: ['gene_id', 'gene_version', 'gene_name', 'gene_source', 'gene_biotype', 'transcript_id', 'transcript_version', 'transcript_name', 'transcript_source', 'transcript_biotype', 'tag', 'transcript_support_level', 'exon_number', 'exon_id', 'exon_version', 'protein_id', 'protein_version', 'ccds_id']


,seqname,source,feature,start,end,score,strand,frame,gene_id,gene_version,...,transcript_source,transcript_biotype,tag,transcript_support_level,exon_number,exon_id,exon_version,protein_id,protein_version,ccds_id
0,1,havana,gene,108344807,108347562,NaN,+,0,ENSMUSG00000104478,2,...,,,,,,,,,,
1,1,havana,transcript,108344807,108347562,NaN,+,0,ENSMUSG00000104478,2,...,havana,TEC,"gencode_basic,gencode_primary,Ensembl_canonical",NA,,,,,,
2,1,havana,exon,108344807,108347562,NaN,+,0,ENSMUSG00000104478,2,...,havana,TEC,"gencode_basic,gencode_primary,Ensembl_canonical",NA,1,ENSMUSE00001337335,2,,,
3,1,havana,gene,6980784,6981446,NaN,+,0,ENSMUSG00000104385,2,...,,,,,,,,,,
4,1,havana,transcript,6980784,6981446,NaN,+,0,ENSMUSG00000104385,2,...,havana,processed_pseudogene,"gencode_basic,gencode_primary,Ensembl_canonical",NA,,,,,,


In [12]:
tf_name_df = pd.read_csv(tf_name_file, sep="\t")
tf_names = tf_name_df["TF_Name"].unique()
print(len(tf_names))

1513


Downloads or loads ChIP-Atlas peaks for each TF in the TF names file from all cell types for a given organism.

In [6]:
if not Path(DATA_DIR / "ground_truth_files" / "chip_atlas_mm10_all_sample.csv").exists():
    print("Creating sampled ChIP-Atlas data...")
    chip_atlas_df = pd.read_csv(DATA_DIR / "ground_truth_files" / "chip_atlas_mm10_all.csv")
    chip_atlas_df.sample(frac = 0.05, random_state=42).to_csv(DATA_DIR / "ground_truth_files" / "chip_atlas_mm10_all_sample.csv", index=False)
else:
    print("Loading sampled ChIP-Atlas data...")
    chip_atlas_df = pd.read_csv(DATA_DIR / "ground_truth_files" / "chip_atlas_mm10_all_sample.csv")

chip_atlas_df = chip_atlas_df.rename(columns={"source_id":"gene_id"})
chip_atlas_tfs = chip_atlas_df["gene_id"].unique()
print(f"TFs with ChIP-Atlas data: {len(chip_atlas_tfs)}, Total peaks: {chip_atlas_df['peak_id'].nunique()}")
display(chip_atlas_df.head())

Loading sampled ChIP-Atlas data...
TFs with ChIP-Atlas data: 443, Total peaks: 7743973


,gene_id,peak_id
0,Ctcf,chr1:93029293-93029685
1,Nfib,chr2:26447512-26447670
2,Spi1,chr10:26725328-26725513
3,Egr2,chr13:46619756-46619917
4,Ctcf,chr7:125783202-125783599


In [ ]:
# mm10_chip_atlas_file = DATA_DIR / "ground_truth_files" / "chipatlas_mESC.csv"
# chip_atlas_df = pd.read_csv(mm10_chip_atlas_file)
# display(chip_atlas_df.head())

,gene_id,peak_id
0,Smad4,chr1:3003564-3003922
1,Ctcf,chr1:3012605-3012815
2,Ctcf,chr1:3012635-3012824
3,Epitope,chr1:3031387-3031654
4,Smad4,chr1:3031454-3031677


Download all FASTA sequences for ChIP-Atlas mm10 TFs

In [ ]:
email = "luminarada@gmail.com"
gene_names = chip_atlas_tfs
organism = "mouse"
output_dir = PROJECT_DIR / "data" / "tf_data" / "tf_sequences"
output_dir.mkdir(parents=True, exist_ok=True)

saved_files = utils.download_gene_protein_fastas(
    gene_names=gene_names,
    organism=organism,
    output_dir=output_dir,
    email=email,
)

[1/443] Saved Arid5a: NP_001165676.1 (590 aa)
[2/443] Saved Arid3a: NP_001275554.1 (601 aa)
[3/443] Saved Mnt: NP_034943.3 (591 aa)
[4/443] Saved Tfap2a: NP_035677.3 (439 aa)
[5/443] Saved Tfap4: NP_112459.1 (338 aa)
[6/443] Saved Hmga2: NP_034571.1 (108 aa)
[7/443] Saved Myf5: NP_032682.1 (255 aa)
[8/443] Saved Mlxipl: NP_067430.2 (864 aa)
[9/443] Saved Kdm5b: NP_690855.2 (1544 aa)
[10/443] Saved Arnt: NP_001032826.1 (791 aa)
[11/443] Saved Arid2: NP_780460.3 (1828 aa)
[12/443] Saved Ncoa1: NP_001395531.1 (1447 aa)
[13/443] Saved Npas3: NP_001357891.1 (951 aa)
[14/443] Saved Ahr: NP_038492.1 (805 aa)
[15/443] Saved Srebf1: NP_035610.1 (1134 aa)
[16/443] Saved Srebf2: NP_150087.1 (1130 aa)
[17/443] Saved Hes1: NP_032261.1 (282 aa)
[18/443] Saved Tfap2c: NP_033361.2 (449 aa)
[19/443] Saved Hif1a: NP_034561.2 (836 aa)
[20/443] Saved Tfeb: NP_001155194.1 (475 aa)
[21/443] Saved Ascl2: NP_032580.2 (263 aa)
[22/443] Saved Ptf1a: NP_061279.2 (324 aa)
[23/443] Saved Ncoa2: NP_032704.2 (1462 a

### 5. TF Protein Structure Information

> NOTE: This portion is from TFBindFormer

I can then use foldseek to create 3Di FASTA files from the TF amino acid FASTA files. 

First, we will create the a foldseek database of 3di tokens using the **ProstT5** protein language model. This encodes protein folding predictions into a sequence form.
```bash
sbatch generate_3di_tokens.sh
```

Next, we combine the 3di structural information with protein amino acid sequence information, separated by unique identifier tokens. These are then projected to the size of the model.
```bash
sbatch extract_tf_embeddings.sh
```

In [11]:
arid5_protein_embedding = torch.load(PROJECT_DIR / "data" / "tf_data" / "tf_embeddings" / "Arid5a_protein_embedding.pt")
print(f"Arid5a protein embedding shape: {arid5_protein_embedding.shape}")

Arid5a protein embedding shape: torch.Size([590, 128])


/tmp/ipykernel_3050219/3202318718.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  arid5_protein_embedding = torch.load(PROJECT_DIR / "data" / "tf_data" / "tf_embeddings"

### 6. Prior Knowledge Network Information

I also have prior knowledge about which genes activate other genes from KEGG and Reactome, along with protein-protein interaction information from STRING

In [12]:
tf_names_upper = [tf.upper() for tf in tf_names]
dataset_tfs = rna_pseudobulk.index.intersection(tf_names_upper).to_list()
print(f"TFs in dataset: {len(dataset_tfs)}")
dataset_tgs = rna_pseudobulk.index.to_list()
print(f"Genes in dataset: {len(dataset_tgs)}")

TFs in dataset: 214
Genes in dataset: 2453


In [13]:
kegg_pkn = DATA_DIR / "prior_knowledge_network_data" / "mm10" / "KEGG" / "kegg_mm10_pkn.csv"
string_pkn = DATA_DIR / "prior_knowledge_network_data" / "mm10" / "STRING" / "string_mm10_pkn.csv"
trrust_pkn = DATA_DIR / "prior_knowledge_network_data" / "mm10" / "TRRUST" / "trrust_mm10_pkn.csv"

kegg_df = pd.read_csv(kegg_pkn)
kegg_df = kegg_df[kegg_df["TF"].isin(dataset_tfs) & kegg_df["TG"].isin(dataset_tgs)]
kegg_df = kegg_df[kegg_df["kegg_signal"] != 0]

string_df = pd.read_csv(string_pkn)
string_df["TF"] = string_df["TF"].str.upper()
string_df["TG"] = string_df["TG"].str.upper()
string_df = string_df[string_df["TF"].isin(dataset_tfs) & string_df["TG"].isin(dataset_tgs)]
high_confidence_string_df = string_df[string_df["string_combined_score"].quantile(0.75) <= string_df["string_combined_score"]]

trrust_df = pd.read_csv(trrust_pkn)
trrust_df = trrust_df[trrust_df["TF"].isin(dataset_tfs) & trrust_df["TG"].isin(dataset_tgs)]
trrust_df = trrust_df[trrust_df["trrust_sign"] != 0]

print("KEGG interactions:")
print(f"  - Unique TFs: {kegg_df['TF'].nunique()}")
print(f"  - Unique TGs: {kegg_df['TG'].nunique()}")
print(f"  - Total interactions: {len(kegg_df)}")
display(kegg_df)

print("\nSTRING interactions:")
print(f"  - Unique TFs: {high_confidence_string_df['TF'].nunique()}")
print(f"  - Unique TGs: {high_confidence_string_df['TG'].nunique()}")
print(f"  - Total interactions: {len(high_confidence_string_df)}")
display(high_confidence_string_df.head())

print("\nTRRUST interactions:")
print(f"  - Unique TFs: {trrust_df['TF'].nunique()}")
print(f"  - Unique TGs: {trrust_df['TG'].nunique()}")
print(f"  - Total interactions: {len(trrust_df)}")


display(trrust_df)

KEGG interactions:
  - Unique TFs: 35
  - Unique TGs: 88
  - Total interactions: 217


,TF,TG,kegg_signal,kegg_n_pathways,kegg_pathways
26817,MECOM,MAPK10,-1,1,mmu04010
37988,MEF2C,KLF2,1,2,"mmu04371,mmu05418"
38337,JUND,MMP9,1,1,mmu04657
38364,JUND,CALCR,1,1,mmu04380
38365,JUND,ITGB3,1,1,mmu04380
...,...,...,...,...,...
82839,NR3C2,SGK1,1,1,mmu04960
82842,NR3C2,SCNN1A,1,1,mmu04960
82846,NR3C2,ATP1B1,1,1,mmu04960
82849,NR3C2,FXYD2,1,1,mmu04960



STRING interactions:
  - Unique TFs: 44
  - Unique TGs: 60
  - Total interactions: 123


,TF,TG,string_neighborhood_score,string_fusion_score,string_cooccurence_score,string_coexpression_score,string_experimental_score,string_database_score,string_textmining_score,string_combined_score
1672,SOX9,GATA4,0,0,0,0,42,900,683,0.966
7966,FOSB,FOS,0,0,0,829,102,900,985,0.999
7969,FOSB,JUND,0,0,0,177,819,900,998,0.999
7971,FOSB,FOSL1,0,0,0,187,94,900,986,0.998
7975,FOSB,JUN,0,0,0,604,762,900,997,0.999



TRRUST interactions:
  - Unique TFs: 103
  - Unique TGs: 172
  - Total interactions: 289


,TF,TG,trrust_sign,trrust_regulation,trrust_pmids,trrust_support_n
63,AR,SCNN1A,1,Activation,16172422,1
64,AR,SOX2,-1,Repression,23326489,1
160,BACH2,PRDM1,-1,Repression,17046816;21296099,1
168,BCL11A,FRZB,1,Activation,22491945,1
194,BCL6,BCL6,-1,Repression,12407182,1
...,...,...,...,...,...,...
6470,ZFPM1,GATA3,-1,Repression,11971000,1
6474,ZFPM2,GATA4,-1,Repression,12606418,1
6482,ZIC2,AFP,-1,Repression,16765502,1
6483,ZIC2,EPHA4,1,Activation,24360543,1


---

## How can I test the model?

### Subset training by excluding chromosomes
I can test on a subset of chromosomes, where I only use TGs from chromosomes 1-15 for training and reserve 16-18 for validation and 19 for testing (or some other split like that).

I can use the gene TSS reference DataFrame to get a map of which genes should be in each train/val/test set.

In [7]:
gene_chrom = gene_ref_df[["seqname", "gene_name"]].rename(columns={"seqname": "chrom", "gene_name": "TG"})
gene_chrom["chrom"] = gene_chrom["chrom"]
gene_chrom["TG"] = gene_chrom["TG"]
# gene_chrom = gene_chrom[gene_chrom["TG"].str.upper().isin(dataset_tgs)]

# Train set: genes on chromosomes 1 - 15
train_genes = gene_chrom[gene_chrom["chrom"].isin([str(i) for i in range(1, 16)])]
print(f"Train set: {len(train_genes)} genes")

# Validation set: genes on chromosomes 16 - 18
val_genes = gene_chrom[gene_chrom["chrom"].isin([str(i) for i in range(16, 19)])]
print(f"Validation set: {len(val_genes)} genes")

# Test set: genes on chromosome 19
test_genes = gene_chrom[gene_chrom["chrom"].isin([str(19)])]
print(f"Test set: {len(test_genes)} genes")

Train set: 2029954 genes
Validation set: 271611 genes
Test set: 78083 genes


---

## Training dataset

The PKN doesn't really have a good number of edges. I might be able to use it as an input to the classifier model, but I don't think that it will be a good way to train the model. 

I can train the model on ChIP-seq peaks for the genes and peaks on the train chromosomes and see how well it predicts edges for the validation and test edges.

### Inputs
- TF identity embedding
- TF expression
- TF structure/sequence embedding
- TG identity embedding
- TG expression
- ATAC accessibility
- ATAC peak to TG distance
- ATAC peak DNA sequence embedding

## TF to DNA binding Model
First, I want to determine if the TF will bind to the DNA. This will depend on the TF identity, the TF structural information, and the ATAC peak DNA sequence embedding.

For training, I can use the TF-peak binding data from ChIP-Atlas. I will use the `utils.create_true_false_edges()` function to generate a balanced dataset of True edges (TF-DNA binding interactions IN the dataset) and False edges (TF-DNA binding interactions NOT in the dataset). The `utils.create_true_false_edges()` function creates False edges by assigning DNA binding sites to the wrong TF.

### Create True/False labeled dataset from ChIP-Atlas

In [13]:
test_genes["TG"].unique()

array(['Gm22271', 'Klf9', 'Mir1192', ..., 'Myrf', 'Dagla', 'Gm30042'],
      shape=(2009,), dtype=object)

Only keep TFs that we have embeddings for

In [8]:
tf_embedding_dir = PROJECT_DIR / "data" / "tf_data" / "tf_embeddings"
tf_embedding_files = list(tf_embedding_dir.glob("*_protein_embedding.pt"))
embedded_tf_names = [f.stem.split("_protein_embedding")[0] for f in tf_embedding_files]
print(f"TFs with embeddings: {len(embedded_tf_names)}")
print(f"Example TFs with embeddings: {embedded_tf_names[:10]}")

TFs with embeddings: 476
Example TFs with embeddings: ['Lhx3', 'Zfp57', 'Foxk1', 'Erg', 'Yy1', 'Zfp281', 'Foxa3', 'Npm1', 'Klf3', 'Arid3a']


In [15]:
# Only keep interactions for TFs that we have embeddings for
chip_atlas_df = chip_atlas_df[chip_atlas_df["gene_id"].isin(embedded_tf_names)]

# chip_atlas_df = chip_atlas_df[chip_atlas_df['gene_id'].isin(test_genes['TG'].unique())]
mm10_tf_names = chip_atlas_df["gene_id"].unique().tolist() # Original non-uppercase TF names

chip_atlas_df.head()

chip_seq_peaks = chip_atlas_df["peak_id"].unique().tolist()

print(chip_atlas_df["gene_id"].nunique())

443


Create a labeled dataset of true and false edges. False edges are created by shuffling peaks to incorrect TFs.

In [18]:
true_interactions, false_interactions = utils.create_true_false_edges(
    edge_df=chip_atlas_df,
    tf_names=mm10_tf_names,
    tf_col="gene_id",
    item_col="peak_id",
    pct_true_edges=0.25,
    true_false_ratio=0.5
)

print(f"True interactions: {len(true_interactions)} e.g. {list(true_interactions)[0]}")
print(f"False interactions: {len(false_interactions)} e.g. {list(false_interactions)[0]}")

Generating sampled unobserved TF-item edges: 100%|██████████████████████████████| 968737/968737 [00:00<00:00, 1298834.83it/s]


True interactions: 1937474 e.g. ('Cebpa', 'chr12:85637607-85637954')
False interactions: 968737 e.g. ('Hnf4g', 'chr10:43702996-43703197')


In [19]:
def create_labeled_tf_peak_dataset(
    true_interactions: set[tuple[str, str]],
    false_interactions: set[tuple[str, str]],
    tf_name_to_idx: dict[str, int],
    peak_id_to_idx: dict[str, int],
    drop_missing: bool = True,
) -> pd.DataFrame:
    """
    Create a labeled TF-peak interaction dataset.

    Labels:
        true interactions  -> 1
        false interactions -> 0

    Returns
    -------
    pd.DataFrame with columns:
        tf_name, peak_id, tf_idx, peak_idx, label
    """

    rows = []

    for tf, peak in true_interactions:
        rows.append((tf, peak, 1))

    for tf, peak in false_interactions:
        rows.append((tf, peak, 0))

    df = pd.DataFrame(rows, columns=["tf_name", "peak_id", "label"])

    df["tf_idx"] = df["tf_name"].map(tf_name_to_idx)
    df["peak_idx"] = df["peak_id"].map(peak_id_to_idx)

    missing_mask = df["tf_idx"].isna() | df["peak_idx"].isna()

    if missing_mask.any():
        n_missing = missing_mask.sum()

        if drop_missing:
            print(f"Dropping {n_missing} interactions with missing TF or peak indices.")
            df = df.loc[~missing_mask].copy()
        else:
            missing_examples = df.loc[missing_mask].head()
            raise ValueError(
                f"{n_missing} interactions are missing TF or peak indices.\n"
                f"Examples:\n{missing_examples}"
            )

    df["tf_idx"] = df["tf_idx"].astype(np.int64)
    df["peak_idx"] = df["peak_idx"].astype(np.int64)
    df["label"] = df["label"].astype(np.float32)

    # Optional: shuffle rows
    df = df.sample(frac=1.0, random_state=123).reset_index(drop=True)

    return df

Create a name to index map of the TFs and ChIP-seq peak locations. These are used to create TF, peak, and label tensors that can easily be mapped back to the peak ID and gene names.

In [20]:
# Map the TF names and peak IDs to their respective indices
tf_name_to_idx = {tf: idx for idx, tf in enumerate(mm10_tf_names)}
peak_id_to_idx = {peak: idx for idx, peak in enumerate(chip_seq_peaks)}

# Create a labeled dataset of TF-peak interactions
tf_peak_labeled_df = create_labeled_tf_peak_dataset(
    true_interactions=true_interactions,
    false_interactions=false_interactions,
    tf_name_to_idx=tf_name_to_idx,
    peak_id_to_idx=peak_id_to_idx,
    drop_missing=False,
)

# Extract the TF indices, peak indices, and labels as numpy arrays for model input
edge_tf_idx = tf_peak_labeled_df["tf_idx"].to_numpy(dtype=np.int64)
edge_peak_idx = tf_peak_labeled_df["peak_idx"].to_numpy(dtype=np.int64)
edge_labels = tf_peak_labeled_df["label"].to_numpy(dtype=np.float32)

# Convert to PyTorch tensors
edge_tf_idx_tensor = torch.as_tensor(edge_tf_idx, dtype=torch.long)
edge_peak_idx_tensor = torch.as_tensor(edge_peak_idx, dtype=torch.long)
edge_labels_tensor = torch.as_tensor(edge_labels, dtype=torch.float32)

tf_peak_labeled_df.head()

,tf_name,peak_id,label,tf_idx,peak_idx
0,Spi1,chr12:112632901-112633376,1.0,2,1446031
1,Hoxc13,chr16:34661603-34662130,1.0,177,3598846
2,Lhx3,chrX:159963322-159963557,0.0,14,719552
3,Nr4a1,chr2:165778970-165779220,0.0,139,3199208
4,Ctcf,chr19:29059732-29060103,1.0,0,2185300


### Create TF protein sequence/structure embedding

#### Download the FASTA sequences for the TFs in the ChIP-seq dataset

In [ ]:
# email = "luminarada@gmail.com"
# gene_names = chip_atlas_df["gene_id"].unique().tolist()  # Original non-uppercase TF names
# organism = "mouse"
# output_dir = PROJECT_DIR / "data" / "tf_data" / "tf_sequences"
# output_dir.mkdir(parents=True, exist_ok=True)

# saved_files = utils.download_gene_protein_fastas(
#     gene_names=gene_names,
#     organism=organism,
#     output_dir=output_dir,
#     email=email,
# )

[1/3] No records found for Epitope
[2/3] No records found for GFP
[3/3] No records found for Biotin


#### 2. Generate TF embeddings with TFBindFormer

```bash
sbatch bash_scripts/generate_3di_tokens.sh
```

and

```bash
sbatch bash_scripts/extract_tf_embeddings.sh
```

This will give us the protein sequence and structural information for each TF in the dataset.

#### 3. Pad protein length to the longest protein so they can be joined into one dataset

Load in all of the TF protein embeddings in the order that they appear in the TF to index mapping.

In [21]:
from torch.nn.utils.rnn import pad_sequence

def load_ordered_tf_embeddings(
    embedding_dir,
    tf_name_to_idx,
    suffix="_protein_embedding.pt",
    weights_only=False,
):
    embedding_dir = Path(embedding_dir)

    # Map TF name -> file path
    available_files = {}

    for path in embedding_dir.glob(f"*{suffix}"):
        tf_name = path.name.replace(suffix, "")
        available_files[tf_name] = path

    n_tfs = len(tf_name_to_idx)

    ordered_tf_names = [None] * n_tfs
    ordered_embeddings = [None] * n_tfs
    ordered_lengths = [0] * n_tfs

    missing_tfs = []

    for tf_name, tf_idx in tf_name_to_idx.items():
        ordered_tf_names[tf_idx] = tf_name

        if tf_name not in available_files:
            missing_tfs.append(tf_name)
            continue

        emb = torch.load(
            available_files[tf_name],
            weights_only=weights_only,
            map_location="cpu",
        )

        # Convert [1, L, D] -> [L, D]
        if emb.ndim == 3 and emb.shape[0] == 1:
            emb = emb.squeeze(0)

        emb = emb.float()

        ordered_embeddings[tf_idx] = emb
        ordered_lengths[tf_idx] = emb.shape[0]

    if len(missing_tfs) > 0:
        raise FileNotFoundError(
            f"Missing embeddings for {len(missing_tfs)} TFs. "
            f"Examples: {missing_tfs[:20]}"
        )

    lengths = torch.tensor(ordered_lengths, dtype=torch.long)

    embeddings_padded = pad_sequence(
        ordered_embeddings,
        batch_first=True,
        padding_value=0.0,
    )

    max_len = embeddings_padded.shape[1]

    mask = torch.arange(max_len).unsqueeze(0) < lengths.unsqueeze(1)

    return {
        "tf_names": ordered_tf_names,
        "embeddings": embeddings_padded,  # [n_tfs, max_tf_len, embedding_dim]
        "lengths": lengths,              # [n_tfs]
        "mask": mask,                    # [n_tfs, max_tf_len]
    }

In [22]:
embedding_dir = PROJECT_DIR / "data" / "tf_data" / "tf_embeddings"

# Load all of the TF embeddings in the order specified by tf_name_to_idx
tf_data = load_ordered_tf_embeddings(
    embedding_dir=embedding_dir,
    tf_name_to_idx=tf_name_to_idx,
)

tf_embeddings_tensor: torch.Tensor = tf_data["embeddings"]
tf_mask_tensor: torch.Tensor = tf_data["mask"]
tf_lengths_tensor: torch.Tensor = tf_data["lengths"]
tf_names_ordered: list[str] = tf_data["tf_names"]

### Create centered peak onehot arrays

Create one-hot encoded DNA sequences of size 2 * `flank_size`. The center of the peak is used to find the `flank_size` upstream and downstream nucleotides.

In [42]:
importlib.reload(utils)

<module 'utils' from '/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/dev/notebooks/simple_model_testing/utils.py'>

In [43]:
peak_ids = list(tf_peak_labeled_df["peak_id"].unique())

chrom_sizes = utils.load_chrom_sizes(chrom_sizes_path)

peak_onehot_cache_path = PROJECT_DIR / "data" / "training_data_cache_notebook" / "peak_onehot_array.pt"
if not peak_onehot_cache_path.parent.exists():
    peak_onehot_cache_path.parent.mkdir(parents=True, exist_ok=True)

if os.path.exists(peak_onehot_cache_path):
    print(f"Loading cached peak tensor from: {peak_onehot_cache_path}")
    peak_tensor = torch.load(peak_onehot_cache_path)
else:
    print("Cache not found. Creating peak one-hot array...")
    peak_onehot_array = utils.create_centered_peak_onehot_array(
        peak_ids=peak_ids,
        genome_fasta=genome_fasta_path,
        chrom_sizes=chrom_sizes,
        peak_id_to_idx=peak_id_to_idx,
        flank_size=64,
        dtype=np.uint8,
        pad_out_of_bounds=True,
        show_progress=True,
        num_workers=10,
        chunk_size=1000,
    )
    peak_tensor = torch.as_tensor(peak_onehot_array, dtype=torch.uint8)
    torch.save(peak_tensor, peak_onehot_cache_path)

Cache not found. Creating peak one-hot array...


One-hot peaks:   0%|                                                                            | 0/2620312 [0…

The TF tensor, TF mask, and peak tensors shapes are:

In [44]:
print(f"TF embeddings tensor shape: (n_tfs, max_tf_len, embedding_dim)")
print(f"  - {tf_embeddings_tensor.shape}")
print(f"TF mask tensor shape: (n_tfs, max_tf_len)")
print(f"  - {tf_mask_tensor.shape}")
print(f"Peak tensor shape: (n_peaks, peak_len, num_nucleotides)")
print(f"  - {peak_tensor.shape}")

TF embeddings tensor shape: (n_tfs, max_tf_len, embedding_dim)
  - torch.Size([443, 5588, 128])
TF mask tensor shape: (n_tfs, max_tf_len)
  - torch.Size([443, 5588])
Peak tensor shape: (n_peaks, peak_len, num_nucleotides)
  - torch.Size([7743973, 128, 4])


### PyTorch Dataset

In [52]:
class TFPeakEdgeDataset(Dataset):
    def __init__(
        self,
        tf_embeddings,
        tf_mask,
        peak_embeddings,
        edge_tf_idx,
        edge_peak_idx,
        edge_labels,
    ):
        # Inputs
        self.tf_embeddings = tf_embeddings
        self.tf_mask = tf_mask
        self.peak_embeddings = peak_embeddings

        # Labels and indices for edges
        self.edge_tf_idx = edge_tf_idx
        self.edge_peak_idx = edge_peak_idx
        self.edge_labels = edge_labels

    def __len__(self):
        return len(self.edge_labels)

    def __getitem__(self, idx):
        tf_idx = self.edge_tf_idx[idx]
        peak_idx = self.edge_peak_idx[idx]

        return {
            "tf_embedding": self.tf_embeddings[tf_idx].float(),         # [max_tf_len, 128]
            "tf_mask": self.tf_mask[tf_idx],                            # [max_tf_len]
            "peak_embedding": self.peak_embeddings[peak_idx].float(),   # [512, 4]
            "label": self.edge_labels[idx],                             # scalar
            "tf_idx": tf_idx,
            "peak_idx": peak_idx,
        }

In [53]:
edge_dataset = TFPeakEdgeDataset(
    tf_embeddings=tf_embeddings_tensor,
    tf_mask=tf_mask_tensor,
    peak_embeddings=peak_tensor,
    edge_tf_idx=edge_tf_idx_tensor,
    edge_peak_idx=edge_peak_idx_tensor,
    edge_labels=edge_labels_tensor,
)

### Train/Validation DataLoaders

In [109]:
# Chromosome-stratified split by peak_id
def _get_chrom(peak_id: str) -> str:
    # Expect format like "chr1:123-456"
    return peak_id.split(":", 1)[0]

edge_peak_ids = tf_peak_labeled_df["peak_id"].to_numpy()
edge_chroms = np.array([_get_chrom(pid) for pid in edge_peak_ids])

# Split the dataset into train/val/test sets by chromosome
# Test set: chromosomes 1-15
# Val set: chromosomes 16-17
# Test set: chromosomes 18 and 19
train_chroms = {f"chr{i}" for i in range(1, 16)}
val_chroms = {f"chr{i}" for i in range(16, 18)}
test_chroms = {"chr18", "chr19"}

# Create boolean masks for train/val/test edges based on their chromosome
train_mask = np.isin(edge_chroms, list(train_chroms))
val_mask = np.isin(edge_chroms, list(val_chroms))
test_mask = np.isin(edge_chroms, list(test_chroms))

# Get the indices of the edges in each split
train_idx = np.where(train_mask)[0]
val_idx = np.where(val_mask)[0]
test_idx = np.where(test_mask)[0]

# Subset the datasets by the split indices
train_dataset = Subset(edge_dataset, train_idx)
val_dataset = Subset(edge_dataset, val_idx)
test_dataset = Subset(edge_dataset, test_idx)

# Create dataloaders for each split
train_loader = DataLoader(
    train_dataset,
    batch_size=512,
    shuffle=True,
    num_workers=7,
    pin_memory=False,
    prefetch_factor=2,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=7,
    pin_memory=False,
    prefetch_factor=2,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=512,
    shuffle=False,
    num_workers=7,
    pin_memory=False,
    prefetch_factor=2,
)

### TF to DNA Binding Prediction Model

Uses the `bidirectional-cross-attention` package from https://github.com/lucidrains/bidirectional-cross-attention.

In [28]:
import models.tf_to_dna as tf_to_dna_module
importlib.reload(tf_to_dna_module)

<module 'models.tf_to_dna' from '/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/dev/notebooks/simple_model_testing/models/tf_to_dna.py'>

Build a new `TFPeakBindingModel` and the PyTorch Lightning model wrapper for training

In [110]:
base_model = tf_to_dna_module.TFPeakBindingModel(
    tf_embedding_dim=128,
    hidden_dim=128,
    dropout=0.1,
    num_layers=2,
    num_heads=4,
    dim_head=32,
)
base_model.train()

# PyTorch Lightning wrapper for training
lit_model = tf_to_dna_module.LitTFPeakBindingModel(
    model=base_model,
    lr=1e-4,
    weight_decay=1e-4,
    pos_weight=None,
)

In [111]:
import pytorch_lightning as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor, TQDMProgressBar
from lightning.pytorch.loggers import WandbLogger

run_name = "tfbind_train_notebook_v1"
output_dir = PROJECT_DIR / "checkpoints" / run_name

checkpoint_callback = ModelCheckpoint(
    dirpath=output_dir,
    filename="epoch_{epoch:02d}",
    monitor="val/auroc",
    mode="max",
    save_top_k=3,
    save_last=True,
    auto_insert_metric_name=False,
)

early_stopping_callback = EarlyStopping(
    monitor="val/loss",
    mode="min",
    patience=10,
)

lr_monitor = LearningRateMonitor(logging_interval="epoch")

wandb_logger = WandbLogger(
    project="tf_peak_binding",
    name=run_name,
    save_dir=output_dir
)

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    precision="16-mixed",
    logger=wandb_logger,
    callbacks=[
        checkpoint_callback,
        early_stopping_callback,
        lr_monitor,
    ],
    gradient_clip_val=1.0,
    gradient_clip_algorithm="norm",
    log_every_n_steps=100,
    default_root_dir=output_dir,
    enable_progress_bar=True,
    enable_checkpointing=True,
    check_val_every_n_epoch=1,
)

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python ...
INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:light

In [114]:
wandb_logger.experiment.finish()

epoch,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
lr-AdamW,▁▁
train/acc,▁
train/auprc,▁
train/auroc,▁
train/loss_epoch,▁
train/loss_step,█▆▆▆▅▅▄▄▄▅▅▄▄▃▃▂▃▂▄▃▂▂▂▅▃▂▂▃▁▂▂▂▂▂▂▂▁▁▂▁
trainer/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
val/acc,▁
val/auprc,▁
+4,...


In [112]:
# Free cuda memory
torch.cuda.empty_cache()
import gc
gc.collect()

20

In [113]:
torch.set_float32_matmul_precision('medium')

trainer.fit(
    lit_model,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)
wandb_logger.experiment.finish()

INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/dev/notebooks/simple_model_testing/checkpoints/tfbind_train_notebook_v1 exists and is not empty.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model       │ TFPeakBindingModel     │  959 K │ train │     0 │
│ 1 │ train_auroc │ BinaryAUROC            │      0 │ train │     0 │
│ 2 │ val_auroc   │ BinaryAUROC            │      0 │ train │     0 │
│ 3 │ train_auprc │ BinaryAveragePrecision │      0 │ train │     0 │
│ 4 │ val_auprc   │ BinaryAveragePrecision │      0 │ train │     0 │
│ 5 │ train_acc   │ BinaryAccuracy         │      0 │ train │     0 │
│ 6 │ val_acc     │ BinaryAccuracy         │      0 │ train │     0 │
└───┴─────────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 959 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 959 K                                                                                                
Total estimated model params size (MB): 3                                                                          
Modules in train mode: 84                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


### Profiling the TF-DNA prediction model

In [117]:
def _move_batch_to_device(batch, device):
    moved = {
        "tf_embedding": batch["tf_embedding"].to(device),
        "tf_mask": batch["tf_mask"].to(device),
        "peak_embedding": batch["peak_embedding"].to(device),
        "label": batch["label"].to(device),
        "tf_idx": batch["tf_idx"].to(device),
        "peak_idx": batch["peak_idx"].to(device),
    }

    return moved

def profile_training_step_times_bag(
    model,
    loader,
    optimizer,
    criterion,
    device,
    n_batches=10,
):
    import time
    import torch

    model.train()

    prev = time.perf_counter()

    for batch_idx, batch in enumerate(loader):
        if batch_idx >= n_batches:
            break

        t_loaded = time.perf_counter()

        batch = _move_batch_to_device(batch, device)

        if device.type == "cuda":
            torch.cuda.synchronize()
        t_h2d = time.perf_counter()

        labels = batch["label"]

        optimizer.zero_grad(set_to_none=True)

        logits = model.forward(
            tf_embedding=batch["tf_embedding"],
            tf_mask=batch["tf_mask"],
            peak_embedding=batch["peak_embedding"]
        )

        loss = criterion(logits, labels)

        if device.type == "cuda":
            torch.cuda.synchronize()
        t_forward = time.perf_counter()

        loss.backward()
        optimizer.step()

        if device.type == "cuda":
            torch.cuda.synchronize()
        t_backward = time.perf_counter()

        load_time = t_loaded - prev
        h2d_time = t_h2d - t_loaded
        forward_time = t_forward - t_h2d
        backward_time = t_backward - t_forward
        step_time = t_backward - prev

        print(
            f"batch {batch_idx:03d} | "
            f"load={load_time:.4f}s | "
            f"h2d={h2d_time:.4f}s | "
            f"forward={forward_time:.4f}s | "
            f"backward+opt={backward_time:.4f}s | "
            f"step={step_time:.4f}s | "
            f"loss={loss.item():.4f}"
        )

        prev = time.perf_counter()

print("Loading new TFTGBindingModel")
profile_model = tf_to_dna_module.TFPeakBindingModel(
    tf_embedding_dim=128,
    hidden_dim=128,
    dropout=0.1,
    num_layers=2,
    num_heads=4,
    dim_head=32,
)
profile_model.train()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
profile_model = profile_model.to(device)

criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, profile_model.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)

profile_training_step_times_bag(
    model=profile_model,
    loader=train_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    n_batches=5,
)

Loading new TFTGBindingModel
batch 000 | load=2.1627s | h2d=0.2506s | forward=0.1629s | backward+opt=0.7201s | step=3.2963s | loss=0.6811
batch 001 | load=0.1551s | h2d=0.8028s | forward=0.4429s | backward+opt=0.2847s | step=1.6856s | loss=0.6736
batch 002 | load=0.2059s | h2d=0.1793s | forward=0.9767s | backward+opt=0.2841s | step=1.6459s | loss=0.6640
batch 003 | load=0.1494s | h2d=0.1806s | forward=0.1629s | backward+opt=0.9497s | step=1.4427s | loss=0.6495
batch 004 | load=0.1538s | h2d=0.3842s | forward=0.1629s | backward+opt=0.6469s | step=1.3478s | loss=0.6576


---

## TF-TG Prediction Model

The TF-DNA prediction model uses the TF protein structure, TF protein sequence, and DNA sequence to learn to predict which TFs will bind to open regions of the DNA.

Once I have a model that can accurately predict which TFs bind around a TG, I need to further narrow down which TFs are causally linked to the TG expression. To do this, I can use:

1. TF expression
2. TG expression
3. Peak accessibility
4. Prior knowledge
5. Peak to TG TSS distance

These can be used as evidence by the model to learn when a TF is likely to be regulating a TG when bound to a given DNA location.


I can load my trained TF to DNA binding model from a training checkpoint and inject the frozen weights into the TFTGRegulation model.

#### Function to create a new TF-TG binding model

Uses the frozen weights from a trained TFPeakBindingModel to create a fresh TFTGRegulationModel.

In [13]:
import torch
import models.tf_to_dna as tf_to_dna_module
import models.tf_to_tg as tf_to_tg_module

def create_new_tf_tg_binding_model(ckpt_path: Path) -> tf_to_tg_module.TFTGRegulationModel:
    # 1) Recreate the base TF→DNA model with the same hyperparameters
    base_model = tf_to_dna_module.TFPeakBindingModel(
        tf_embedding_dim=128,
        hidden_dim=128,
        dropout=0.3,
        num_layers=4,
        num_heads=4,
        dim_head=32,
    )

    # 2) Wrap in Lightning module and load checkpoint
    lit_model = tf_to_dna_module.LitTFPeakBindingModel.load_from_checkpoint(
        checkpoint_path=ckpt_path,
        model=base_model,
        lr=1e-4,
        weight_decay=1e-4,
        pos_weight=None,
    )

    state = torch.load(ckpt_path, map_location="cpu")
    lit_model.load_state_dict(state["state_dict"], strict=True)

    # 3) Get the trained base model and freeze it
    trained_tf_peak_model = lit_model.model
    trained_tf_peak_model.eval()
    for p in trained_tf_peak_model.parameters():
        p.requires_grad = False

    # 4) Inject into your TF→TG model
    tf_tg_model = tf_to_tg_module.TFTGRegulationModel(
        pretrained_tf_peak_model=trained_tf_peak_model,
        d_model=128,
    )
    
    return tf_tg_model

ckpt_path = PROJECT_DIR / "checkpoints" / "tfbind_train_3671604" / "epoch=02-val_auroc=0.9054-val_loss=0.2948.ckpt"


#### Load and merge the ground truth datasets

In [14]:
def load_ground_truth(ground_truth_file: Path | str) -> pd.DataFrame:
    if type(ground_truth_file) == str:
        ground_truth_file = Path(ground_truth_file)
        
    print(f"Loading ground truth file: {ground_truth_file.name}")

    if ground_truth_file.suffix == ".csv":
        sep = ","
    elif ground_truth_file.suffix == ".tsv":
        sep="\t"
        
    ground_truth_df = pd.read_csv(ground_truth_file, sep=sep, on_bad_lines="skip", engine="python")
    
    if "chip" in ground_truth_file.name and "atlas" in ground_truth_file.name:
        ground_truth_df = ground_truth_df[["source_id", "target_id"]]

    if ground_truth_df.columns[0] != "Source" or ground_truth_df.columns[1] != "Target":
        ground_truth_df = ground_truth_df.rename(columns={ground_truth_df.columns[0]: "Source", ground_truth_df.columns[1]: "Target"})
    
    # Capitalize TF and TG names for consistency
    ground_truth_df["Source"] = ground_truth_df["Source"].astype(str).str.capitalize()
    ground_truth_df["Target"] = ground_truth_df["Target"].astype(str).str.capitalize()
    
    # Build TF, TG, and edge sets for quick lookup later
    gt = ground_truth_df[["Source", "Target"]].dropna()

    return gt

def load_ground_truth_files(gt_path_list: list[Path]) -> pd.DataFrame:
    gt_dfs = []
    for gt_path in gt_path_list:
        gt_df = load_ground_truth(gt_path)
        gt_dfs.append(gt_df)
    merged_gt_df = pd.concat(gt_dfs, ignore_index=True)
    return merged_gt_df

mm10_chip_atlas_file = DATA_DIR / "ground_truth_files" / "chip_atlas_tf_peak_tg_dist.csv"
rn111_file = DATA_DIR / "ground_truth_files" / "RN111.tsv"
rn112_file = DATA_DIR / "ground_truth_files" / "RN112.tsv"
rn114_file = DATA_DIR / "ground_truth_files" / "RN114.tsv"
rn116_file = DATA_DIR / "ground_truth_files" / "RN116.tsv"

merged_ground_truth_df = load_ground_truth_files([
    mm10_chip_atlas_file,
    rn111_file,
    rn112_file,
    rn114_file,
    rn116_file,
])

Loading ground truth file: chip_atlas_tf_peak_tg_dist.csv
Loading ground truth file: RN111.tsv
Loading ground truth file: RN112.tsv
Loading ground truth file: RN114.tsv
Loading ground truth file: RN116.tsv


#### Subset the ground truth genes to train/val/test by chromosome

We can avoid data leakage by only training for edges with genes from chromosomes 1-15, validate with genes on chromosomes 16-18, and test on genes from chromosome 19. This method allows us to train on the same dataset we use to evaluate model performance. The TF-DNA binding predictor was only trained on peaks from chromosomes 1-15 as well.

Use the gene TSS reference to determine which genes are on test/val/train chromosomes

In [15]:
gene_chrom = gene_ref_df[["seqname", "gene_name"]].rename(columns={"seqname": "chrom", "gene_name": "TG"})
gene_chrom["chrom"] = gene_chrom["chrom"]
gene_chrom["TG"] = gene_chrom["TG"]
# gene_chrom = gene_chrom[gene_chrom["TG"].str.upper().isin(dataset_tgs)]

# Train set: genes on chromosomes 1 - 15
train_genes = gene_chrom[gene_chrom["chrom"].isin([str(i) for i in range(1, 16)])]["TG"].unique()
print(f"Train set: {len(train_genes)} genes")

# Validation set: genes on chromosomes 16 - 18
val_genes = gene_chrom[gene_chrom["chrom"].isin([str(i) for i in range(16, 19)])]["TG"].unique()
print(f"Validation set: {len(val_genes)} genes")

# Test set: genes on chromosome 19
test_genes = gene_chrom[gene_chrom["chrom"].isin([str(19)])]["TG"].unique()
print(f"Test set: {len(test_genes)} genes")

Train set: 62426 genes
Validation set: 7790 genes
Test set: 2009 genes


Subset the genes in the ground truth into train/val/test splits based on chromosome

In [16]:
tf_tg_ground_truth_df = merged_ground_truth_df[["Source", "Target"]]

# Create train/val/test splits of the ground truth based on the TG's chromosome
train_genes_set = {g for g in train_genes}
val_genes_set = {g for g in val_genes}
test_genes_set = {g for g in test_genes}

# Filter the ground truth interactions to only include those where the TG is in the respective split
tf_tg_ground_truth_train_df = tf_tg_ground_truth_df[
    tf_tg_ground_truth_df["Target"].isin(train_genes_set)
].copy()
tf_tg_ground_truth_val_df = tf_tg_ground_truth_df[
    tf_tg_ground_truth_df["Target"].isin(val_genes_set)
].copy()
tf_tg_ground_truth_test_df = tf_tg_ground_truth_df[
    tf_tg_ground_truth_df["Target"].isin(test_genes_set)
].copy()

ground_truth_genes = set(tf_tg_ground_truth_df["Target"].unique())
print(f"Ground truth genes: {len(ground_truth_genes)}")
print(f"Train genes in ground truth: {tf_tg_ground_truth_train_df['Target'].nunique()}")
print(f"Validation genes in ground truth: {tf_tg_ground_truth_val_df['Target'].nunique()}")
print(f"Test genes in ground truth: {tf_tg_ground_truth_test_df['Target'].nunique()}")

Ground truth genes: 34326
Train genes in ground truth: 17309
Validation genes in ground truth: 2208
Test genes in ground truth: 667


#### Create labeled TF-TG dataset

We can use the existing TF mapping to keep the TF names linked to the same indices as in training. We will create a new TG mapping, then use the `create_true_false_edges()` and `create_labeled_tf_tg_dataset()` functions to generate training data for the classifier.

In [17]:
def create_labeled_tf_tg_dataset(
    true_interactions: set[tuple[str, str]],
    false_interactions: set[tuple[str, str]],
    tf_name_to_idx: dict[str, int],
    tg_id_to_idx: dict[str, int],
    drop_missing: bool = True,
) -> pd.DataFrame:
    """
    Create a labeled TF-TG interaction dataset.

    Labels:
        true interactions  -> 1
        false interactions -> 0

    Returns
    -------
    pd.DataFrame with columns:
        tf_name, tg_id, tf_idx, tg_idx, label
    """

    rows = []

    # Create rows for true interactions with label 1
    for tf, tg in true_interactions:
        rows.append((tf, tg, 1))

    # Create rows for false interactions with label 0
    for tf, tg in false_interactions:
        rows.append((tf, tg, 0))

    # Convert to DataFrame
    df = pd.DataFrame(rows, columns=["tf_name", "tg_id", "label"])

    # Map TF names and TG IDs to their respective indices
    df["tf_idx"] = df["tf_name"].map(tf_name_to_idx)
    df["tg_idx"] = df["tg_id"].map(tg_id_to_idx)

    # Check for any missing mappings and handle them
    missing_mask = df["tf_idx"].isna() | df["tg_idx"].isna()

    if missing_mask.any():
        n_missing = missing_mask.sum()

        if drop_missing:
            print(f"Dropping {n_missing} interactions with missing TF or TG indices.")
            df = df.loc[~missing_mask].copy()
        else:
            missing_examples = df.loc[missing_mask].head()
            raise ValueError(
                f"{n_missing} interactions are missing TF or TG indices.\n"
                f"Examples:\n{missing_examples}"
            )

    # Convert indices and labels to appropriate data types
    df["tf_idx"] = df["tf_idx"].astype(np.int64)
    df["tg_idx"] = df["tg_idx"].astype(np.int64)
    df["label"] = df["label"].astype(np.float32)

    # Optional: shuffle rows
    df = df.sample(frac=1.0, random_state=123).reset_index(drop=True)

    return df

training_cache_dir = PROJECT_DIR / "data" / "training_data_cache_new"
tf_name_to_idx_cache_path = training_cache_dir / "tf_name_to_idx.csv"
tf_name_to_idx = pd.read_csv(tf_name_to_idx_cache_path).set_index("tf_name")["tf_idx"].to_dict()

# Create a mapping from TG name to index across all ground-truth TGs
tg_id_to_idx = {tg: idx for idx, tg in enumerate(tf_tg_ground_truth_df["Target"].unique())}

def _create_labeled_df(gt_df: pd.DataFrame, seed: int = 123) -> pd.DataFrame:
    true_edges, false_edges = utils.create_true_false_edges(
        edge_df=gt_df,
        tf_names=tf_name_to_idx.keys(),
        tf_col="Source",
        item_col="Target",
        pct_true_edges=1.0,
        true_false_ratio=2.0,
        seed=seed,
    )
    return create_labeled_tf_tg_dataset(
        true_interactions=true_edges,
        false_interactions=false_edges,
        tf_name_to_idx=tf_name_to_idx,
        tg_id_to_idx=tg_id_to_idx,
        drop_missing=False,
    )

# Create labeled datasets, split by train/val/test chromosomes
tf_tg_labeled_train_df = _create_labeled_df(tf_tg_ground_truth_train_df)
tf_tg_labeled_val_df = _create_labeled_df(tf_tg_ground_truth_val_df)
tf_tg_labeled_test_df = _create_labeled_df(tf_tg_ground_truth_test_df)

print(f"Train labeled edges: {len(tf_tg_labeled_train_df)}")
print(f"Val labeled edges: {len(tf_tg_labeled_val_df)}")
print(f"Test labeled edges: {len(tf_tg_labeled_test_df)}")

Generating sampled unobserved TF-item edges:   0%|                                               | 0/1387154 […

Generating sampled unobserved TF-item edges:   0%|                                                | 0/175574 […

Generating sampled unobserved TF-item edges:   0%|                                                 | 0/53508 […

Train labeled edges: 2080731
Val labeled edges: 263361
Test labeled edges: 80262


#### Prepare the input tensors

To speed things up, we will precompute some of the data and just look it up when building the tensors. Avoids repeated computation.

In [ ]:
dataset_peaks = atac_pseudobulk.index.to_list()
valid_chroms = {f"chr{i}" for i in range(1, 20)}
dataset_peaks = [peak for peak in dataset_peaks if peak.split(":", 1)[0] in valid_chroms]
atac_peak_map = {peak: idx for idx, peak in enumerate(dataset_peaks)}
print(f"Peaks in dataset: {dataset_peaks[:2]}...{dataset_peaks[-2:]} (total: {len(dataset_peaks)})")

# Build TFTGRegulationModel inputs from TF-TG ground truth + ATAC peaks
training_cache_dir = PROJECT_DIR / "data" / "training_data_cache_new"
tf_embedding_cache_path = training_cache_dir / "tf_embeddings.pt"
tf_mask_cache_path = training_cache_dir / "tf_masks.pt"
atac_peak_onehot_cache_path = training_cache_dir / "atac_peak_onehot_array.pt"

tf_embeddings_tensor = torch.load(tf_embedding_cache_path)
tf_mask_tensor = torch.load(tf_mask_cache_path)

# Create or load the one-hot encoded peak sequences for the ATAC peaks in the dataset
dataset_peaks = list(atac_peak_map.keys())
if os.path.exists(atac_peak_onehot_cache_path):
    atac_peak_tensor = torch.load(atac_peak_onehot_cache_path)
else:
    atac_peak_array = utils.create_centered_peak_onehot_array(
        peak_ids=dataset_peaks,
        genome_fasta=genome_fasta_path,
        chrom_sizes=utils.load_chrom_sizes(chrom_sizes_path),
        peak_id_to_idx=atac_peak_map,
        flank_size=64,
        dtype=np.uint8,
        pad_out_of_bounds=True,
        num_workers=10,
        show_progress=True,
        chunk_size=1000,
    )
    atac_peak_tensor = torch.as_tensor(atac_peak_array, dtype=torch.float32)
    torch.save(atac_peak_tensor, atac_peak_onehot_cache_path)
    
if atac_peak_tensor.dtype == torch.uint8:
    atac_peak_tensor = atac_peak_tensor.float()

# Align cell IDs across RNA and ATAC pseudobulk matrices
rna_pseudobulk_norm = rna_pseudobulk.copy()
rna_pseudobulk_norm.index = rna_pseudobulk_norm.index.str.upper()

common_cells = sorted(set(rna_pseudobulk_norm.columns) & set(atac_pseudobulk.columns))
print(f"Common cells: {len(common_cells)}")

peak_to_gene = peak_to_gene_distance.copy()
peak_to_gene["target_id_norm"] = peak_to_gene["target_id"].str.upper()

tg_embedding_table = torch.nn.Embedding(len(tg_id_to_idx), 128)

def prepare_tftg_lookup_tables(
    peak_to_gene,
    atac_peak_map,
    atac_pseudobulk,
    rna_pseudobulk_norm,
    dataset_peaks,
    common_cells,
    max_precompute_peaks=64,
):
    valid_peak_set = set(atac_peak_map.keys())

    peak_to_gene_valid = peak_to_gene[
        peak_to_gene["peak_id"].isin(valid_peak_set)
    ].copy()

    peak_to_gene_valid["abs_dist"] = peak_to_gene_valid["TSS_dist"].abs()

    tg_to_peak_info = {}

    for tg_norm, sub in peak_to_gene_valid.groupby("target_id_norm", sort=False):
        sub = sub.sort_values("abs_dist").head(max_precompute_peaks)

        peak_ids = sub["peak_id"].tolist()
        peak_indices = np.asarray([atac_peak_map[p] for p in peak_ids], dtype=np.int64)
        peak_distances = sub["TSS_dist"].to_numpy(dtype=np.float32)

        tg_to_peak_info[tg_norm] = {
            "peak_ids": peak_ids,
            "peak_indices": peak_indices,
            "peak_distances": peak_distances,
        }

    cell_to_idx = {cell: i for i, cell in enumerate(common_cells)}

    atac_mat = (
        atac_pseudobulk
        .reindex(index=dataset_peaks, columns=common_cells)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    rna_mat = (
        rna_pseudobulk_norm
        .reindex(columns=common_cells)
        .fillna(0.0)
        .to_numpy(dtype=np.float32)
    )

    gene_to_rna_idx = {
        gene: i for i, gene in enumerate(rna_pseudobulk_norm.index)
    }

    return tg_to_peak_info, cell_to_idx, atac_mat, rna_mat, gene_to_rna_idx

tg_to_peak_info, cell_to_idx, atac_mat, rna_mat, gene_to_rna_idx = prepare_tftg_lookup_tables(
    peak_to_gene=peak_to_gene,
    atac_peak_map=atac_peak_map,
    atac_pseudobulk=atac_pseudobulk,
    rna_pseudobulk_norm=rna_pseudobulk_norm,
    dataset_peaks=dataset_peaks,
    common_cells=common_cells,
    max_precompute_peaks=64,
)


Peaks in dataset: ['chr1:3035460-3036350', 'chr1:3044677-3045614']...['chr9:124489545-124490297', 'chr9:124494438-124495224'] (total: 195301)


/tmp/ipykernel_2935366/226342376.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tf_embeddings_tensor = torch.load(tf_embedding_cache_path)
/tmp/ipykernel_2935366/22634

Common cells: 5199


Next we build the TF-TG input tensors. Creates TF-TG data with `max_peak_per_tg` peak expression and distance information for each TG, sorted by distance from the TG. Each pair also gets TF and TG expression data from `max_cells_per_pair` random cells. 

In [27]:
from collections import defaultdict

def build_tftg_inputs(
    tf_tg_df,
    max_peaks_per_tg=64,
    max_cells_per_pair=8,
    seed=123,
    zero_fields=None,
    *,
    tg_to_peak_info,
    cell_to_idx,
    atac_mat,
    rna_mat,
    gene_to_rna_idx,
    common_cells,
    tf_name_to_idx,
    tg_id_to_idx,
):
    """
    Build compact TFTGRegulationModel inputs.

    This version stores indices and small numeric arrays only.
    Large lookup tensors are retrieved lazily in Dataset.__getitem__().
    """

    rng = np.random.default_rng(seed)

    tf_names = []
    tg_names = []
    cell_ids = []
    labels = []

    tf_indices = []
    tg_indices = []
    peak_indices_all = []

    peak_access = []
    peak_dist = []
    peak_masks = []
    tf_expr = []
    tg_expr = []
    peak_id_sets = []

    if zero_fields is None:
        zero_fields = set()
    else:
        zero_fields = set(zero_fields)

    ZEROABLE_FIELDS = {
        "tf_embedding",
        "peak_sequences",
        "peak_accessibility",
        "peak_distance",
        "tf_expression",
        "tg_expression",
        "tg_embedding",
    }

    invalid = zero_fields - ZEROABLE_FIELDS
    if invalid:
        raise ValueError(
            f"Invalid zero_fields: {sorted(invalid)}. "
            f"Valid options are: {sorted(ZEROABLE_FIELDS)}"
        )

    common_cells = list(common_cells)
    n_common_cells = len(common_cells)

    for _, row in tqdm(
        tf_tg_df.iterrows(),
        total=len(tf_tg_df),
        ncols=100,
        desc="Building compact TF-TG inputs",
    ):
        tf_name = row["tf_name"]
        tg_name = row["tg_id"]
        label = float(row["label"])

        tf_norm = str(tf_name).upper()
        tg_norm = str(tg_name).upper()

        tf_idx = tf_name_to_idx.get(tf_name)
        tg_idx = tg_id_to_idx.get(tg_name)

        if tf_idx is None or tg_idx is None:
            continue

        peak_info = tg_to_peak_info.get(tg_norm)
        if peak_info is None:
            continue

        # -----------------------------
        # Peak index / distance setup
        # -----------------------------
        peak_ids_real = list(peak_info["peak_ids"][:max_peaks_per_tg])
        peak_indices_real = list(peak_info["peak_indices"][:max_peaks_per_tg])
        peak_dst_real = list(peak_info["peak_distances"][:max_peaks_per_tg])

        n_peaks = len(peak_indices_real)
        if n_peaks == 0:
            continue

        peak_indices = np.asarray(peak_indices_real, dtype=np.int64)
        peak_dst = np.asarray(peak_dst_real, dtype=np.float32)
        peak_mask = np.ones(n_peaks, dtype=bool)
        peak_ids = list(peak_ids_real)

        if n_peaks < max_peaks_per_tg:
            pad_len = max_peaks_per_tg - n_peaks

            # Use 0 as dummy padded peak index.
            # It will be masked out by peak_mask.
            peak_indices = np.pad(
                peak_indices,
                (0, pad_len),
                constant_values=0,
            )

            peak_dst = np.pad(
                peak_dst,
                (0, pad_len),
                constant_values=0.0,
            )

            peak_mask = np.pad(
                peak_mask,
                (0, pad_len),
                constant_values=False,
            )

            peak_ids = peak_ids + [""] * pad_len

        # -----------------------------
        # Sample cells
        # -----------------------------
        if max_cells_per_pair is None or max_cells_per_pair >= n_common_cells:
            sampled_cells = common_cells
        else:
            sampled_cells = rng.choice(
                common_cells,
                size=max_cells_per_pair,
                replace=False,
            ).tolist()

        sampled_cell_indices = np.asarray(
            [cell_to_idx[c] for c in sampled_cells],
            dtype=np.int64,
        )

        # -----------------------------
        # Vectorized ATAC accessibility
        # -----------------------------
        peak_acc_matrix = np.zeros(
            (len(sampled_cell_indices), max_peaks_per_tg),
            dtype=np.float32,
        )

        peak_acc_matrix[:, :n_peaks] = atac_mat[
            np.ix_(peak_indices_real, sampled_cell_indices)
        ].T

        # -----------------------------
        # Vectorized RNA expression
        # -----------------------------
        tf_rna_idx = gene_to_rna_idx.get(tf_norm)
        tg_rna_idx = gene_to_rna_idx.get(tg_norm)

        if tf_rna_idx is None:
            tf_expr_vals = np.zeros(len(sampled_cell_indices), dtype=np.float32)
        else:
            tf_expr_vals = rna_mat[tf_rna_idx, sampled_cell_indices].astype(np.float32)

        if tg_rna_idx is None:
            tg_expr_vals = np.zeros(len(sampled_cell_indices), dtype=np.float32)
        else:
            tg_expr_vals = rna_mat[tg_rna_idx, sampled_cell_indices].astype(np.float32)

        # -----------------------------
        # Append one row per sampled cell
        # -----------------------------
        for j, cell_id in enumerate(sampled_cells):
            tf_indices.append(tf_idx)
            tg_indices.append(tg_idx)
            peak_indices_all.append(peak_indices)

            peak_access.append(peak_acc_matrix[j])
            peak_dist.append(peak_dst)
            peak_masks.append(peak_mask)

            tf_expr.append(float(tf_expr_vals[j]))
            tg_expr.append(float(tg_expr_vals[j]))

            peak_id_sets.append(peak_ids)
            tf_names.append(tf_name)
            tg_names.append(tg_name)
            cell_ids.append(cell_id)
            labels.append(label)

    if len(labels) == 0:
        raise ValueError(
            "No TF-TG examples were created. Check TF/TG IDs, peak-to-gene mapping, "
            "and overlap with ATAC/RNA matrices."
        )

    return {
        "tf_name": tf_names,
        "tg_name": tg_names,
        "cell_id": cell_ids,
        "peak_ids": peak_id_sets,

        "label": torch.tensor(labels, dtype=torch.float32),

        # Compact indices
        "tf_idx": torch.tensor(tf_indices, dtype=torch.long),
        "tg_idx": torch.tensor(tg_indices, dtype=torch.long),
        "peak_indices": torch.tensor(np.stack(peak_indices_all), dtype=torch.long),

        # Small per-cell/per-edge tensors
        "peak_accessibility": torch.tensor(np.stack(peak_access), dtype=torch.float32),
        "peak_mask": torch.tensor(np.stack(peak_masks), dtype=torch.bool),
        "peak_distance": torch.tensor(np.stack(peak_dist), dtype=torch.float32),
        "tf_expression": torch.tensor(tf_expr, dtype=torch.float32),
        "tg_expression": torch.tensor(tg_expr, dtype=torch.float32),
    }
    
def _sample_df(df: pd.DataFrame, n: int | None, seed: int) -> pd.DataFrame:
    if n is None or len(df) <= n:
        return df
    return df.sample(n=n, random_state=seed)


# Sample a subset of the TF-TG pairs for faster training during development
sample_pairs = 5000
print(f"Sampling {sample_pairs} TF-TG pairs for train/val/test subsets...")
print(f"Original train pairs: {len(tf_tg_labeled_train_df)}")

tf_tg_train_subset = _sample_df(tf_tg_labeled_train_df, n=sample_pairs, seed=123)
tf_tg_val_subset = _sample_df(tf_tg_labeled_val_df, n=sample_pairs // 2, seed=123)
tf_tg_test_subset = _sample_df(tf_tg_labeled_test_df, n=sample_pairs // 4, seed=123)

zero_fields = None

common_build_kwargs = dict(
    max_peaks_per_tg=8,
    max_cells_per_pair=16,
    zero_fields=zero_fields,
    tg_to_peak_info=tg_to_peak_info,
    cell_to_idx=cell_to_idx,
    atac_mat=atac_mat,
    rna_mat=rna_mat,
    gene_to_rna_idx=gene_to_rna_idx,
    common_cells=common_cells,
    tf_name_to_idx=tf_name_to_idx,
    tg_id_to_idx=tg_id_to_idx,
)

tftg_inputs_train = build_tftg_inputs(
    tf_tg_train_subset,
    seed=123,
    **common_build_kwargs,
)

tftg_inputs_val = build_tftg_inputs(
    tf_tg_val_subset,
    seed=124,
    **common_build_kwargs,
)

tftg_inputs_test = build_tftg_inputs(
    tf_tg_test_subset,
    seed=125,
    **common_build_kwargs,
)

print(
    {
        "train": len(tftg_inputs_train["label"]),
        "val": len(tftg_inputs_val["label"]),
        "test": len(tftg_inputs_test["label"]),
    }
)

Sampling 5000 TF-TG pairs for train/val/test subsets...
Original train pairs: 2080731


Building compact TF-TG inputs:   0%|                                       | 0/5000 [00:00<?, ?it/s]

Building compact TF-TG inputs:   0%|                                       | 0/2500 [00:00<?, ?it/s]

Building compact TF-TG inputs:   0%|                                       | 0/1250 [00:00<?, ?it/s]

{'train': 60640, 'val': 29856, 'test': 16160}


#### TFTGEdgeBagDataset PyTorch Dataset

We use a Pytorch Dataset class to handle passing the data to the model during training. 

In [28]:


class TFTGEdgeBagDataset(Dataset):
    def __init__(
        self,
        inputs,
        *,
        tf_embeddings_tensor,
        tf_mask_tensor,
        atac_peak_tensor,
        tg_embedding_table,
        max_cells_per_pair=None,
        zero_fields=None,
        strict=True,
    ):
        """
        Groups rows from build_tftg_inputs() by (tf_name, tg_name).

        Each item is one TF-TG edge bag containing multiple sampled cells.
        Large tensors are gathered lazily using compact indices.
        """

        self.inputs = inputs
        self.tf_embeddings_tensor = tf_embeddings_tensor
        self.tf_mask_tensor = tf_mask_tensor
        self.atac_peak_tensor = atac_peak_tensor
        self.tg_embedding_table = tg_embedding_table
        self.max_cells_per_pair = max_cells_per_pair

        if zero_fields is None:
            zero_fields = set()
        else:
            zero_fields = set(zero_fields)

        self.zero_fields = zero_fields

        required_list_keys = [
            "tf_name",
            "tg_name",
            "cell_id",
        ]

        required_tensor_keys = [
            "label",
            "tf_idx",
            "tg_idx",
            "peak_indices",
            "peak_accessibility",
            "peak_distance",
            "peak_mask",
            "tf_expression",
            "tg_expression",
        ]

        lengths = {}

        for key in required_list_keys:
            lengths[key] = len(inputs[key])

        for key in required_tensor_keys:
            lengths[key] = inputs[key].shape[0]

        unique_lengths = sorted(set(lengths.values()))

        if len(unique_lengths) != 1:
            msg = "\n".join(
                [f"{key:20s}: {length}" for key, length in lengths.items()]
            )

            if strict:
                raise ValueError(
                    "Input fields have inconsistent first-dimension lengths:\n"
                    f"{msg}\n\n"
                    "This usually means one of the lists was appended without "
                    "a matching tensor row, or tensors were filtered after metadata "
                    "was created."
                )
            else:
                self.n_rows = min(unique_lengths)
                print(
                    "WARNING: Input fields have inconsistent lengths. "
                    f"Using first {self.n_rows} rows only.\n{msg}"
                )
        else:
            self.n_rows = unique_lengths[0]

        groups = defaultdict(list)

        for i in range(self.n_rows):
            tf = inputs["tf_name"][i]
            tg = inputs["tg_name"][i]
            groups[(tf, tg)].append(i)

        self.edge_keys = list(groups.keys())
        self.groups = [
            torch.tensor(v, dtype=torch.long)
            for v in groups.values()
        ]

    def __len__(self):
        return len(self.groups)

    def _maybe_zero(self, name, tensor):
        if name in self.zero_fields:
            return torch.zeros_like(tensor)
        return tensor

    def __getitem__(self, idx):
        row_idx = self.groups[idx]

        if self.max_cells_per_pair is not None:
            row_idx = row_idx[: self.max_cells_per_pair]

        row_idx_list = row_idx.tolist()

        label = self.inputs["label"][row_idx[0]]

        # These should be constant within a TF-TG edge bag
        tf_idx = self.inputs["tf_idx"][row_idx[0]]
        tg_idx = self.inputs["tg_idx"][row_idx[0]]

        # Gather large static lookup tensors once per edge bag
        tf_embedding = self.tf_embeddings_tensor[tf_idx]
        tf_mask = self.tf_mask_tensor[tf_idx]

        # TG embedding from embedding table
        # Keep this on CPU for DataLoader collation.
        with torch.no_grad():
            tg_embedding = self.tg_embedding_table(
                tg_idx.to(self.tg_embedding_table.weight.device).view(1)
            ).squeeze(0).detach().cpu()

        # Small per-cell tensors
        peak_indices = self.inputs["peak_indices"][row_idx]          # [C, P]
        peak_accessibility = self.inputs["peak_accessibility"][row_idx]  # [C, P]
        peak_distance = self.inputs["peak_distance"][row_idx]        # [C, P]
        peak_mask = self.inputs["peak_mask"][row_idx]                # [C, P]
        tf_expression = self.inputs["tf_expression"][row_idx]        # [C]
        tg_expression = self.inputs["tg_expression"][row_idx]        # [C]

        # Gather peak sequences lazily
        # atac_peak_tensor is probably uint8: [n_peaks, seq_len, 4]
        # peak_sequences: [C, P, seq_len, 4]
        peak_sequences = self.atac_peak_tensor[peak_indices]

        # Repeat static TF/TG features across cells because your current collate/model
        # expect a cell dimension on every field.
        n_cells = row_idx.shape[0]

        tf_embedding = tf_embedding.unsqueeze(0).expand(n_cells, -1, -1)
        tf_mask = tf_mask.unsqueeze(0).expand(n_cells, -1)
        tg_embedding = tg_embedding.unsqueeze(0).expand(n_cells, -1)

        item = {
            "label": label,
            "tf_name": self.edge_keys[idx][0],
            "tg_name": self.edge_keys[idx][1],
            "cell_ids": [
                self.inputs["cell_id"][i]
                for i in row_idx_list
            ],

            "tf_idx": tf_idx,
            "tg_idx": tg_idx,
            "peak_indices": peak_indices,

            "tf_embedding": self._maybe_zero(
                "tf_embedding",
                tf_embedding.float(),
            ),
            "tf_mask": tf_mask.bool(),

            "peak_sequences": self._maybe_zero(
                "peak_sequences",
                peak_sequences.float(),
            ),
            "peak_accessibility": self._maybe_zero(
                "peak_accessibility",
                peak_accessibility.float(),
            ),
            "peak_distance": self._maybe_zero(
                "peak_distance",
                peak_distance.float(),
            ),
            "peak_mask": peak_mask.bool(),

            "tf_expression": self._maybe_zero(
                "tf_expression",
                tf_expression.float(),
            ),
            "tg_expression": self._maybe_zero(
                "tg_expression",
                tg_expression.float(),
            ),
            "tg_embedding": self._maybe_zero(
                "tg_embedding",
                tg_embedding.float(),
            ),
        }

        return item
        
def collate_tftg_edge_bags(batch):
    """
    Pads each TF-TG edge bag to the max number of cells in the batch.
    """

    labels = torch.stack([b["label"] for b in batch]).float()  # [E]

    max_cells = max(b["tf_expression"].shape[0] for b in batch)
    batch_size = len(batch)

    cell_mask = torch.zeros(batch_size, max_cells, dtype=torch.bool)

    output = {
        "label": labels,
        "cell_mask": cell_mask,
        "tf_name": [b["tf_name"] for b in batch],
        "tg_name": [b["tg_name"] for b in batch],
        "cell_ids": [b["cell_ids"] for b in batch],
        "tf_idx": torch.stack([b["tf_idx"] for b in batch]).long(),
        "tg_idx": torch.stack([b["tg_idx"] for b in batch]).long(),
    }

    tensor_keys = [
        "tf_embedding",
        "tf_mask",
        "peak_sequences",
        "peak_accessibility",
        "peak_distance",
        "peak_mask",
        "tf_expression",
        "tg_expression",
        "tg_embedding",
        "peak_indices",
    ]

    for key in tensor_keys:
        example = batch[0][key]
        padded_shape = (batch_size, max_cells) + tuple(example.shape[1:])
        padded = example.new_zeros(padded_shape)

        for i, b in enumerate(batch):
            n_cells = b[key].shape[0]
            padded[i, :n_cells] = b[key]

        output[key] = padded

    for i, b in enumerate(batch):
        n_cells = b["tf_expression"].shape[0]
        cell_mask[i, :n_cells] = True

    return output
        
train_dataset = TFTGEdgeBagDataset(
    tftg_inputs_train,
    tf_embeddings_tensor=tf_embeddings_tensor,
    tf_mask_tensor=tf_mask_tensor,
    atac_peak_tensor=atac_peak_tensor,
    tg_embedding_table=tg_embedding_table,
    max_cells_per_pair=None,
    zero_fields=zero_fields,
    strict=True,
)

val_dataset = TFTGEdgeBagDataset(
    tftg_inputs_val,
    tf_embeddings_tensor=tf_embeddings_tensor,
    tf_mask_tensor=tf_mask_tensor,
    atac_peak_tensor=atac_peak_tensor,
    tg_embedding_table=tg_embedding_table,
    max_cells_per_pair=None,
    zero_fields=zero_fields,
    strict=True,
)

test_dataset = TFTGEdgeBagDataset(
    tftg_inputs_test,
    tf_embeddings_tensor=tf_embeddings_tensor,
    tf_mask_tensor=tf_mask_tensor,
    atac_peak_tensor=atac_peak_tensor,
    tg_embedding_table=tg_embedding_table,
    max_cells_per_pair=None,
    zero_fields=zero_fields,
    strict=True,
)


In [30]:

batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,   # use 0 while debugging
    pin_memory=True,
    collate_fn=collate_tftg_edge_bags,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_tftg_edge_bags,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_tftg_edge_bags,
)


Quick sanity checks to make sure that all of the items have the expected number of items and max indices

In [32]:
for name, inputs in {
    "train": tftg_inputs_train,
    "val": tftg_inputs_val,
    "test": tftg_inputs_test,
}.items():
    print(name)
    print("  n rows:", len(inputs["label"]))
    print("  tf_idx max:", inputs["tf_idx"].max().item())
    print("  tg_idx max:", inputs["tg_idx"].max().item())
    print("  peak_idx max:", inputs["peak_indices"].max().item())

    assert inputs["tf_idx"].min().item() >= 0
    assert inputs["tf_idx"].max().item() < tf_embeddings_tensor.shape[0]

    assert inputs["tg_idx"].min().item() >= 0
    assert inputs["tg_idx"].max().item() < tg_embedding_table.num_embeddings

    assert inputs["peak_indices"].min().item() >= 0
    assert inputs["peak_indices"].max().item() < atac_peak_tensor.shape[0]

train
  n rows: 60640
  tf_idx max: 432
  tg_idx max: 34257
  peak_idx max: 195250
val
  n rows: 29856
  tf_idx max: 432
  tg_idx max: 34237
  peak_idx max: 93594
test
  n rows: 16160
  tf_idx max: 432
  tg_idx max: 33761
  peak_idx max: 99364


**Shapes:**
- **label**: `[num_edges]`
- **cell_mask**: `[num_edges, num_cells]`
- **tf_name**: `[num_edges]`
- **tg_name**: `[num_edges]`
- **cell_ids**: `[num_cells]`
- **tf_embedding**: `[num_edges, num_cells, tf_len, tf_embedding_dim]`
- **tf_mask**: `[num_edges, num_cells, tf_len]`
- **peak_sequences**: `[num_edges, num_cells, num_peaks_per_tg, peak_length, num_nucleotides]`
- **peak_accessibility**: `[num_edges, num_cells, num_peaks_per_tg]`
- **peak_distance**: `[num_edges, num_cells, num_peaks_per_tg]`
- **tf_expression**: `[num_edges, num_cells]`
- **tg_expression**: `[num_edges, num_cells]`
- **tg_embedding**: `[num_edges, num_cells, tg_embedding_size]`
- **peak_mask**: `[num_edges, num_cells, num_peaks_per_tg]`

In [33]:
first_batch = next(iter(train_loader))

print(f"Total Batches: {len(train_loader)}")
for key, value in first_batch.items():
    if isinstance(value, torch.Tensor):
        print(f"{key}: {value.shape} {value.dtype}")
    else:
        print(f"{key}: {type(value)} len={len(value)} example={value[0] if len(value) > 0 else None}")

Total Batches: 474
label: torch.Size([8]) torch.float32
cell_mask: torch.Size([8, 16]) torch.bool
tf_name: <class 'list'> len=8 example=Hand2
tg_name: <class 'list'> len=8 example=Dock1
cell_ids: <class 'list'> len=8 example=['CTGATCACACACCAAC-1', 'AACCTCCTCGCTCACT-1', 'GTCTAGCCATTCCTCG-1', 'GTATTGATCAAGTGTC-1', 'GAGCTAGCAAGGTCCT-1', 'CCCAAACCATTCAGCA-1', 'CGTGACATCGAAGCGG-1', 'GTGTCCAAGTTCCTGC-1', 'GGCATTAGTCCGCTGT-1', 'ACTTGTAAGTTAGGCT-1', 'TAAACAGCATTAGGCC-1', 'ACCGCAATCACTTTAC-1', 'CCTTCAATCCACAATA-1', 'TATAACCCACTAAGTT-1', 'TGCTTAAAGGGCTTTG-1', 'GCCTACTTCGTTACAA-1']
tf_idx: torch.Size([8]) torch.int64
tg_idx: torch.Size([8]) torch.int64
tf_embedding: torch.Size([8, 16, 5588, 128]) torch.float32
tf_mask: torch.Size([8, 16, 5588]) torch.bool
peak_sequences: torch.Size([8, 16, 8, 128, 4]) torch.float32
peak_accessibility: torch.Size([8, 16, 8]) torch.float32
peak_distance: torch.Size([8, 16, 8]) torch.float32
peak_mask: torch.Size([8, 16, 8]) torch.bool
tf_expression: torch.Size([8, 

### Saving the training data to a cache

In [ ]:
import json

TRAINING_CACHE_DIR = (
    PROJECT_DIR
    / "data"
    / "training_data_cache_notebook"
    / "tftg_regulation_flank64_uint8_maxpeaks8_maxcells16"
)

TRAINING_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Save compact split inputs
torch.save(tftg_inputs_train, TRAINING_CACHE_DIR / "tftg_inputs_train.pt")
torch.save(tftg_inputs_val, TRAINING_CACHE_DIR / "tftg_inputs_val.pt")
torch.save(tftg_inputs_test, TRAINING_CACHE_DIR / "tftg_inputs_test.pt")

# Save lookup tensors
torch.save(tf_embeddings_tensor, TRAINING_CACHE_DIR / "tf_embeddings_tensor.pt")
torch.save(tf_mask_tensor, TRAINING_CACHE_DIR / "tf_mask_tensor.pt")
torch.save(atac_peak_tensor, TRAINING_CACHE_DIR / "atac_peak_tensor.pt")

# Save mapping dictionaries and metadata
metadata = {
    "tf_name_to_idx": tf_name_to_idx,
    "tg_id_to_idx": tg_id_to_idx,
    "gene_to_rna_idx": gene_to_rna_idx,
    "cell_to_idx": cell_to_idx,
    "max_peaks_per_tg": 8,
    "max_cells_per_pair": 16,
    "flank_size": 64,
    "peak_dtype": "uint8",
}
with open(TRAINING_CACHE_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

# Save a manifest to keep track of model settings and dataset versions
manifest = {
    "max_peaks_per_tg": 8,
    "max_cells_per_pair": 16,
    "flank_size": 64,
    "atac_peak_tensor_dtype": str(atac_peak_tensor.dtype),
    "atac_peak_tensor_shape": list(atac_peak_tensor.shape),
    "tf_embeddings_tensor_shape": list(tf_embeddings_tensor.shape),
    "tf_mask_tensor_shape": list(tf_mask_tensor.shape),
    "n_train_rows": int(len(tftg_inputs_train["label"])),
    "n_val_rows": int(len(tftg_inputs_val["label"])),
    "n_test_rows": int(len(tftg_inputs_test["label"])),
}

with open(TRAINING_CACHE_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)


### Loading cached data

In [ ]:
# Load the compact split inputs
tftg_inputs_train = torch.load(TRAINING_CACHE_DIR / "tftg_inputs_train.pt")
tftg_inputs_val = torch.load(TRAINING_CACHE_DIR / "tftg_inputs_val.pt")
tftg_inputs_test = torch.load(TRAINING_CACHE_DIR / "tftg_inputs_test.pt")

# Load the lookup tensors
tf_embeddings_tensor = torch.load(training_cache_dir / "tf_embeddings.pt")
tf_mask_tensor = torch.load(training_cache_dir / "tf_masks.pt")
atac_peak_tensor = torch.load(TRAINING_CACHE_DIR / "atac_peak_tensor.pt")

# Load the metadata
with open(TRAINING_CACHE_DIR / "metadata.json", "r") as f:
    metadata = json.load(f)
    
tf_name_to_idx = metadata["tf_name_to_idx"]
tg_id_to_idx = metadata["tg_id_to_idx"]

# Load the manifest and verify tensor shapes and dtypes match expectations
with open(TRAINING_CACHE_DIR / "manifest.json") as f:
    manifest = json.load(f)

print(json.dumps(manifest, indent=2))

assert tuple(manifest["atac_peak_tensor_shape"]) == tuple(atac_peak_tensor.shape)
assert manifest["atac_peak_tensor_dtype"] == str(atac_peak_tensor.dtype)

# Re-create the datasets and dataloaders using the loaded compact inputs and lookup tensors
train_dataset = TFTGEdgeBagDataset(
    tftg_inputs_train,
    tf_embeddings_tensor=tf_embeddings_tensor,
    tf_mask_tensor=tf_mask_tensor,
    atac_peak_tensor=atac_peak_tensor,
    tg_embedding_table=tg_embedding_table,
    zero_fields=zero_fields,
    strict=True,
)

val_dataset = TFTGEdgeBagDataset(
    tftg_inputs_val,
    tf_embeddings_tensor=tf_embeddings_tensor,
    tf_mask_tensor=tf_mask_tensor,
    atac_peak_tensor=atac_peak_tensor,
    tg_embedding_table=tg_embedding_table,
    zero_fields=zero_fields,
    strict=True,
)

test_dataset = TFTGEdgeBagDataset(
    tftg_inputs_test,
    tf_embeddings_tensor=tf_embeddings_tensor,
    tf_mask_tensor=tf_mask_tensor,
    atac_peak_tensor=atac_peak_tensor,
    tg_embedding_table=tg_embedding_table,
    zero_fields=zero_fields,
    strict=True,
)

/tmp/ipykernel_2935366/3042222559.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tftg_inputs_train = torch.load(TRAINING_CACHE_DIR / "tftg_inputs_train.pt")
/tmp/ipyker

{
  "max_peaks_per_tg": 8,
  "max_cells_per_pair": 16,
  "flank_size": 64,
  "atac_peak_tensor_dtype": "torch.float32",
  "atac_peak_tensor_shape": [
    195301,
    128,
    4
  ],
  "tf_embeddings_tensor_shape": [
    443,
    5588,
    128
  ],
  "tf_mask_tensor_shape": [
    443,
    5588
  ],
  "n_train_rows": 60640,
  "n_val_rows": 29856,
  "n_test_rows": 16160
}


In [138]:
importlib.reload(tf_to_tg_module)

<module 'models.tf_to_tg' from '/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/dev/notebooks/simple_model_testing/models/tf_to_tg.py'>

### Model Training

#### Functions for training/evaluation

In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
)

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

def _move_batch_to_device(batch, device):
    moved = {
        "tf_embedding": batch["tf_embedding"].to(device),
        "tf_mask": batch["tf_mask"].to(device),
        "peak_sequences": batch["peak_sequences"].to(device),
        "peak_accessibility": batch["peak_accessibility"].to(device),
        "peak_distance": batch["peak_distance"].to(device),
        "tf_expression": batch["tf_expression"].to(device),
        "tg_expression": batch["tg_expression"].to(device),
        "tg_embedding": batch["tg_embedding"].to(device),
        "label": batch["label"].to(device),
    }

    if "cell_mask" in batch:
        moved["cell_mask"] = batch["cell_mask"].to(device)

    if "peak_mask" in batch:
        moved["peak_mask"] = batch["peak_mask"].to(device)

    return moved

@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
    pooling_mode: str = "lse",
    pooling_temperature: float = 1.0,
):
    model.eval()

    total_loss = 0.0
    n_edges = 0

    for batch in loader:
        batch = _move_batch_to_device(batch, device)

        labels = batch["label"]
        cell_mask = batch["cell_mask"]
        E, C = cell_mask.shape

        edge_logits, cell_logits = model.forward(
            tf_embedding=batch["tf_embedding"],
            tf_mask=batch["tf_mask"],
            peak_sequences=batch["peak_sequences"],
            peak_accessibility=batch["peak_accessibility"],
            peak_distance=batch["peak_distance"],
            tf_expression=batch["tf_expression"],
            tg_expression=batch["tg_expression"],
            peak_mask=batch.get("peak_mask", None),
            cell_mask=cell_mask,
            pooling_mode=pooling_mode,
            pooling_temperature=pooling_temperature,
        )

        loss = criterion(edge_logits, labels)

        total_loss += loss.item() * E
        n_edges += E

    return total_loss / max(n_edges, 1)

def compute_binary_classification_metrics(
    labels,
    scores,
    score_threshold: float = 0.5,
    random_state: int = 42,
):
    """
    labels: array-like of 0/1 labels
    scores: array-like of predicted probabilities after sigmoid
    """

    labels = np.asarray(labels).astype(int).ravel()
    scores = np.asarray(scores).astype(float).ravel()

    preds = (scores >= score_threshold).astype(int)

    accuracy = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, zero_division=0)

    if len(np.unique(labels)) < 2:
        auroc = np.nan
        auprc = np.nan
        rand_auroc = np.nan
        rand_auprc = np.nan
    else:
        auroc = roc_auc_score(labels, scores)
        auprc = average_precision_score(labels, scores)

        rng = np.random.default_rng(random_state)
        rand_scores = rng.permutation(scores)

        rand_auroc = roc_auc_score(labels, rand_scores)
        rand_auprc = average_precision_score(labels, rand_scores)

    return {
        "auroc": auroc,
        "auprc": auprc,
        "rand_auroc": rand_auroc,
        "rand_auprc": rand_auprc,
        "accuracy": accuracy,
        "precision": precision,
        "n_edges": len(labels),
        "n_pos": int(labels.sum()),
        "n_neg": int((labels == 0).sum()),
        "score_threshold": score_threshold,
    }
    
@torch.no_grad()
def evaluate_with_metrics(
    model,
    loader,
    criterion,
    device,
    score_threshold: float = 0.5,
    random_state: int = 42,
    pooling_mode: str = "lse",
    pooling_temperature: float = 1.0,
):
    model.eval()

    total_loss = 0.0
    n_edges = 0

    all_scores = []
    all_labels = []

    for batch in loader:
        batch = _move_batch_to_device(batch, device)

        labels = batch["label"]          # [E]
        cell_mask = batch["cell_mask"]   # [E, C]
        E, C = cell_mask.shape

        edge_logits, cell_logits = model.forward(
            tf_embedding=batch["tf_embedding"],
            tf_mask=batch["tf_mask"],
            peak_sequences=batch["peak_sequences"],
            peak_accessibility=batch["peak_accessibility"],
            peak_distance=batch["peak_distance"],
            tf_expression=batch["tf_expression"],
            tg_expression=batch["tg_expression"],
            peak_mask=batch.get("peak_mask", None),
            cell_mask=cell_mask,
            pooling_mode=pooling_mode,
            pooling_temperature=pooling_temperature,
        )

        loss = criterion(edge_logits, labels)

        total_loss += loss.item() * E
        n_edges += E

        edge_probs = torch.sigmoid(edge_logits)

        all_scores.append(edge_probs.detach().cpu().numpy().ravel())
        all_labels.append(labels.detach().cpu().numpy().ravel())

    mean_loss = total_loss / max(n_edges, 1)

    all_scores = np.concatenate(all_scores)
    all_labels = np.concatenate(all_labels)

    metrics = compute_binary_classification_metrics(
        labels=all_labels,
        scores=all_scores,
        score_threshold=score_threshold,
        random_state=random_state,
    )

    metrics["loss"] = mean_loss
    metrics["score_min"] = float(all_scores.min())
    metrics["score_max"] = float(all_scores.max())
    metrics["score_mean"] = float(all_scores.mean())
    metrics["score_std"] = float(all_scores.std())
    metrics["n_pred_pos"] = int((all_scores >= score_threshold).sum())

    return metrics

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    pooling_mode: str = "lse",
    pooling_temperature: float = 1.0,
):
    model.train()

    total_loss = 0.0
    n_edges = 0

    pbar = tqdm(loader, desc="Training", ncols=250)

    for batch in pbar:
        batch = _move_batch_to_device(batch, device)

        labels = batch["label"]          # [E]
        cell_mask = batch["cell_mask"]   # [E, C]
        E, C = cell_mask.shape

        optimizer.zero_grad(set_to_none=True)

        edge_logits, cell_logits = model.forward(
            tf_embedding=batch["tf_embedding"],
            tf_mask=batch["tf_mask"],
            peak_sequences=batch["peak_sequences"],
            peak_accessibility=batch["peak_accessibility"],
            peak_distance=batch["peak_distance"],
            tf_expression=batch["tf_expression"],
            tg_expression=batch["tg_expression"],
            peak_mask=batch.get("peak_mask", None),
            cell_mask=cell_mask,
            pooling_mode=pooling_mode,
            pooling_temperature=pooling_temperature,
        )

        loss = criterion(edge_logits, labels)

        loss.backward()
        optimizer.step()

        batch_loss = loss.item()
        total_loss += batch_loss * E
        n_edges += E

    return total_loss / max(n_edges, 1)

def train_one_epoch_track_test_metrics(
    model,
    train_loader,
    test_loader,
    optimizer,
    criterion,
    device,
    epoch: int,
    global_step_start: int = 0,
    max_batches: int = 50,
    score_threshold: float = 0.5,
    random_state: int = 42,
    eval_every_n_batches: int = 1,
    pooling_mode: str = "lse",
    pooling_temperature: float = 1.0,
):
    """
    Trains for one epoch using edge-bag pooling.

    Each batch contains E TF-TG edges and C sampled cells per edge.
    The model predicts [E*C] cell logits, then pools them to [E] edge logits.
    """

    model.train()

    total_train_loss = 0.0
    n_train_edges = 0

    metric_rows = []
    global_step = global_step_start

    for batch_num, batch in enumerate(train_loader):
        if batch_num >= max_batches:
            break

        model.train()

        batch = _move_batch_to_device(batch, device)

        edge_logits, cell_logits = model(
            tf_embedding=batch["tf_embedding"],
            tf_mask=batch["tf_mask"],
            peak_sequences=batch["peak_sequences"],
            peak_accessibility=batch["peak_accessibility"],
            peak_distance=batch["peak_distance"],
            tf_expression=batch["tf_expression"],
            tg_expression=batch["tg_expression"],
            cell_mask=batch["cell_mask"],
            peak_mask=batch.get("peak_mask", None),
            pooling_mode="lse",
            pooling_temperature=1.0,
        )

        loss = criterion(edge_logits, batch["label"])

        loss.backward()
        optimizer.step()

        train_batch_loss = loss.item()

        E = edge_logits.shape[0]
        C = batch["cell_mask"].shape[1]
        total_train_loss += train_batch_loss * E
        n_train_edges += E

        global_step += 1

        should_eval = (
            eval_every_n_batches is not None
            and eval_every_n_batches > 0
            and ((batch_num + 1) % eval_every_n_batches == 0)
        )

        if should_eval:
            test_seed = random_state + global_step

            test_metrics = evaluate_with_metrics(
                model=model,
                loader=test_loader,
                criterion=criterion,
                device=device,
                score_threshold=score_threshold,
                random_state=test_seed,
            )

            row = {
                "epoch": epoch,
                "batch_num": batch_num,
                "global_step": global_step,
                "train_batch_loss": train_batch_loss,
                "test_loss": test_metrics["loss"],
                "auroc": test_metrics["auroc"],
                "auprc": test_metrics["auprc"],
                "rand_auroc": test_metrics["rand_auroc"],
                "rand_auprc": test_metrics["rand_auprc"],
                "accuracy": test_metrics["accuracy"],
                "precision": test_metrics["precision"],
                "n_edges": test_metrics["n_edges"],
                "n_pos": test_metrics["n_pos"],
                "n_neg": test_metrics["n_neg"],
                "score_threshold": test_metrics["score_threshold"],
                "E": E,
                "C": C,
                "pooling_mode": pooling_mode,
                "pooling_temperature": pooling_temperature,
            }

            metric_rows.append(row)

            print(
                f"Epoch {epoch:02d} | "
                f"batch {batch_num:04d} | "
                f"step {global_step:05d} | "
                f"E={E} | C={C} | "
                f"train_batch_loss={train_batch_loss:.4f} | "
                f"test_loss={test_metrics['loss']:.4f} | "
                f"AUROC={test_metrics['auroc']:.4f} | "
                f"AUPRC={test_metrics['auprc']:.4f} | "
                f"rand_AUROC={test_metrics['rand_auroc']:.4f} | "
                f"rand_AUPRC={test_metrics['rand_auprc']:.4f} | "
                f"accuracy={test_metrics['accuracy']:.4f} | "
                f"precision={test_metrics['precision']:.4f} | "
                f"pos_rate={test_metrics['n_pos'] / max(test_metrics['n_edges'], 1):.4f}"
            )

    mean_train_loss = total_train_loss / max(n_train_edges, 1)
    metrics_df = pd.DataFrame(metric_rows)

    return mean_train_loss, metrics_df, global_step

#### Evaluate model training per batch

In [ ]:
print("Loading new TFTGBindingModel")
tf_tg_model = create_new_tf_tg_binding_model(ckpt_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tf_tg_model = tf_tg_model.to(device)

criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(tf_tg_model.parameters(), lr=1e-4, weight_decay=1e-4)

num_epochs = 10
global_step = 0

all_metric_dfs = []

initial_metrics = evaluate_with_metrics(
    model=tf_tg_model,
    loader=test_loader,
    criterion=criterion,
    device=device,
    score_threshold=0.5,
    random_state=42,
)

initial_row = {
    "epoch": 0,
    "batch_num": -1,
    "global_step": 0,
    "train_batch_loss": np.nan,
    "test_loss": initial_metrics["loss"],
    "auroc": initial_metrics["auroc"],
    "auprc": initial_metrics["auprc"],
    "rand_auroc": initial_metrics["rand_auroc"],
    "rand_auprc": initial_metrics["rand_auprc"],
    "accuracy": initial_metrics["accuracy"],
    "precision": initial_metrics["precision"],
    "n_edges": initial_metrics["n_edges"],
    "n_pos": initial_metrics["n_pos"],
    "n_neg": initial_metrics["n_neg"],
    "score_threshold": initial_metrics["score_threshold"],
}

print("\nInitial evaluation before training:")
print(initial_row)

print("\nStarting training...")
for epoch in range(1, num_epochs + 1):
    train_loss, epoch_metrics_df, global_step = train_one_epoch_track_test_metrics(
        model=tf_tg_model,
        train_loader=train_loader,
        test_loader=train_loader,  # evaluate on train set for debugging; switch to test_loader for real eval
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        epoch=epoch,
        global_step_start=global_step,
        max_batches=25,
        score_threshold=0.5,
        random_state=42,
        eval_every_n_batches=10,  # evaluate test set after every train batch
    )
    
    

    val_loss = evaluate(tf_tg_model, val_loader, criterion, device)

    all_metric_dfs.append(epoch_metrics_df)

    print(
        f"Epoch {epoch:02d} complete | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f}"
    )

batch_metric_df = pd.concat(all_metric_dfs, ignore_index=True)

batch_metric_path = PROJECT_DIR / "testing_results" / "test_metrics_per_train_batch.csv"
batch_metric_path.parent.mkdir(parents=True, exist_ok=True)

batch_metric_df.to_csv(batch_metric_path, index=False)

batch_metric_df.head()

#### Run model training

In [42]:
print("Loading new TFTGBindingModel")
tf_tg_model = create_new_tf_tg_binding_model(ckpt_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tf_tg_model = tf_tg_model.to(device)

criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, tf_tg_model.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)

num_epochs = 20
score_threshold = 0.5
pooling_mode = "lse"
pooling_temperature = 1.0

epoch_rows = []


def metrics_to_row(
    metrics,
    epoch,
    split,
    train_loss=np.nan,
    lr=np.nan,
):
    pos_rate = metrics["n_pos"] / max(metrics["n_edges"], 1)

    return {
        "epoch": epoch,
        "split": split,
        "train_loss": train_loss,
        "loss": metrics["loss"],
        "auroc": metrics["auroc"],
        "auprc": metrics["auprc"],
        "rand_auroc": metrics["rand_auroc"],
        "rand_auprc": metrics["rand_auprc"],
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "n_edges": metrics["n_edges"],
        "n_pos": metrics["n_pos"],
        "n_neg": metrics["n_neg"],
        "pos_rate": pos_rate,
        "score_threshold": metrics["score_threshold"],
        "lr": lr,
        "pooling_mode": pooling_mode,
        "pooling_temperature": pooling_temperature,
    }
    
print("\nInitial evaluation before training...")

initial_train_metrics = evaluate_with_metrics(
    model=tf_tg_model,
    loader=train_loader,
    criterion=criterion,
    device=device,
    score_threshold=score_threshold,
    random_state=42,
)

initial_val_metrics = evaluate_with_metrics(
    model=tf_tg_model,
    loader=val_loader,
    criterion=criterion,
    device=device,
    score_threshold=score_threshold,
    random_state=43,
)

initial_test_metrics = evaluate_with_metrics(
    model=tf_tg_model,
    loader=test_loader,
    criterion=criterion,
    device=device,
    score_threshold=score_threshold,
    random_state=44,
)

current_lr = optimizer.param_groups[0]["lr"]

epoch_rows.append(
    metrics_to_row(
        metrics=initial_train_metrics,
        epoch=0,
        split="train",
        train_loss=np.nan,
        lr=current_lr,
    )
)

epoch_rows.append(
    metrics_to_row(
        metrics=initial_val_metrics,
        epoch=0,
        split="val",
        train_loss=np.nan,
        lr=current_lr,
    )
)

epoch_rows.append(
    metrics_to_row(
        metrics=initial_test_metrics,
        epoch=0,
        split="test",
        train_loss=np.nan,
        lr=current_lr,
    )
)

print("Initial train:", initial_train_metrics)
print("Initial val:  ", initial_val_metrics)
print("Initial test: ", initial_test_metrics)

print("\nStarting epoch-level training...")

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(
        model=tf_tg_model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        pooling_mode=pooling_mode,
        pooling_temperature=pooling_temperature,
    )

    current_lr = optimizer.param_groups[0]["lr"]

    train_metrics = evaluate_with_metrics(
        model=tf_tg_model,
        loader=train_loader,
        criterion=criterion,
        device=device,
        score_threshold=score_threshold,
        random_state=42 + epoch,
    )

    val_metrics = evaluate_with_metrics(
        model=tf_tg_model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        score_threshold=score_threshold,
        random_state=10_000 + epoch,
    )

    test_metrics = evaluate_with_metrics(
        model=tf_tg_model,
        loader=test_loader,
        criterion=criterion,
        device=device,
        score_threshold=score_threshold,
        random_state=20_000 + epoch,
    )

    epoch_rows.append(
        metrics_to_row(
            metrics=train_metrics,
            epoch=epoch,
            split="train",
            train_loss=train_loss,
            lr=current_lr,
        )
    )

    epoch_rows.append(
        metrics_to_row(
            metrics=val_metrics,
            epoch=epoch,
            split="val",
            train_loss=train_loss,
            lr=current_lr,
        )
    )

    epoch_rows.append(
        metrics_to_row(
            metrics=test_metrics,
            epoch=epoch,
            split="test",
            train_loss=train_loss,
            lr=current_lr,
        )
    )

    print(
        f"Epoch {epoch:03d} | "
        f"train_loss={train_loss:.4f} | "
        f"train_eval_loss={train_metrics['loss']:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"test_loss={test_metrics['loss']:.4f} | "
        f"val_AUROC={val_metrics['auroc']:.4f} | "
        f"val_AUPRC={val_metrics['auprc']:.4f} | "
        f"val_rand_AUROC={val_metrics['rand_auroc']:.4f} | "
        f"val_rand_AUPRC={val_metrics['rand_auprc']:.4f} | "
        f"val_acc={val_metrics['accuracy']:.4f} | "
        f"val_precision={val_metrics['precision']:.4f} | "
        f"val_pos_rate={val_metrics['n_pos'] / max(val_metrics['n_edges'], 1):.4f}"
    )
    
epoch_metric_df = pd.DataFrame(epoch_rows)

epoch_metric_path = PROJECT_DIR / "testing_results" / "metrics_per_epoch.csv"
epoch_metric_path.parent.mkdir(parents=True, exist_ok=True)

epoch_metric_df.to_csv(epoch_metric_path, index=False)

display(epoch_metric_df.head())

import matplotlib.pyplot as plt

plot_df = epoch_metric_df.copy()

for metric in [
    "loss",
    "auroc",
    "auprc",
    "rand_auroc",
    "rand_auprc",
    "accuracy",
    "precision",
    "pos_rate",
]:
    plt.figure(figsize=(7, 5))

    for split, sub in plot_df.groupby("split"):
        plt.plot(sub["epoch"], sub[metric], marker="o", label=split)

    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.title(f"{metric} over epochs")
    plt.legend()
    plt.tight_layout()
    plt.show()

Loading new TFTGBindingModel

Initial evaluation before training...


/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/lightning_fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
/tmp/ipykernel_2935366/2913531

Initial train: {'auroc': 0.6148232551561408, 'auprc': 0.4506613336048979, 'rand_auroc': 0.5029233342117287, 'rand_auprc': 0.3575155215657272, 'accuracy': 0.6401055408970976, 'precision': 0.0, 'n_edges': 3790, 'n_pos': 1364, 'n_neg': 2426, 'score_threshold': 0.5, 'loss': 0.6848302684232868, 'score_min': 0.4823596179485321, 'score_max': 0.48931607604026794, 'score_mean': 0.48505207896232605, 'score_std': 0.0010686684399843216, 'n_pred_pos': 0}
Initial val:   {'auroc': 0.5883899307655269, 'auprc': 0.4179743997825149, 'rand_auroc': 0.49292631626588956, 'rand_auprc': 0.34801586516967153, 'accuracy': 0.6468381564844587, 'precision': 0.0, 'n_edges': 1866, 'n_pos': 659, 'n_neg': 1207, 'score_threshold': 0.5, 'loss': 0.6845204436638476, 'score_min': 0.4824167490005493, 'score_max': 0.4880319833755493, 'score_mean': 0.48507019877433777, 'score_std': 0.0010930141434073448, 'n_pred_pos': 0}
Initial test:  {'auroc': 0.5959152217463388, 'auprc': 0.4328726542056211, 'rand_auroc': 0.5229647347333517, 

Training:   0%|                                                             | 0/474 [00:00<?, ?it/s]

Epoch 001 | train_loss=0.6415 | train_eval_loss=0.6331 | val_loss=0.6342 | test_loss=0.6310 | val_AUROC=0.6135 | val_AUPRC=0.4406 | val_rand_AUROC=0.5020 | val_rand_AUPRC=0.3640 | val_acc=0.6222 | val_precision=0.4538 | val_pos_rate=0.3532


Training:   0%|                                                             | 0/474 [00:00<?, ?it/s]

KeyboardInterrupt: 

#### Profile model training times

In [43]:
def profile_training_step_times_bag(
    model,
    loader,
    optimizer,
    criterion,
    device,
    n_batches=10,
    pooling_mode="lse",
    pooling_temperature=1.0,
):
    import time
    import torch

    model.train()

    prev = time.perf_counter()

    for batch_idx, batch in enumerate(loader):
        if batch_idx >= n_batches:
            break

        t_loaded = time.perf_counter()

        batch = _move_batch_to_device(batch, device)

        if device.type == "cuda":
            torch.cuda.synchronize()
        t_h2d = time.perf_counter()

        labels = batch["label"]
        cell_mask = batch["cell_mask"]
        E, C = cell_mask.shape

        optimizer.zero_grad(set_to_none=True)

        edge_logits, cell_logits = model.forward(
            tf_embedding=batch["tf_embedding"],
            tf_mask=batch["tf_mask"],
            peak_sequences=batch["peak_sequences"],
            peak_accessibility=batch["peak_accessibility"],
            peak_distance=batch["peak_distance"],
            tf_expression=batch["tf_expression"],
            tg_expression=batch["tg_expression"],
            peak_mask=batch.get("peak_mask", None),
            cell_mask=cell_mask,
            pooling_mode=pooling_mode,
            pooling_temperature=pooling_temperature,
        )

        loss = criterion(edge_logits, labels)

        if device.type == "cuda":
            torch.cuda.synchronize()
        t_forward = time.perf_counter()

        loss.backward()
        optimizer.step()

        if device.type == "cuda":
            torch.cuda.synchronize()
        t_backward = time.perf_counter()

        load_time = t_loaded - prev
        h2d_time = t_h2d - t_loaded
        forward_time = t_forward - t_h2d
        backward_time = t_backward - t_forward
        step_time = t_backward - prev

        print(
            f"batch {batch_idx:03d} | "
            f"load={load_time:.4f}s | "
            f"h2d={h2d_time:.4f}s | "
            f"forward={forward_time:.4f}s | "
            f"backward+opt={backward_time:.4f}s | "
            f"step={step_time:.4f}s | "
            f"loss={loss.item():.4f} | "
            f"E={E} C={C}"
        )

        prev = time.perf_counter()

print("Loading new TFTGBindingModel")
tf_tg_model = create_new_tf_tg_binding_model(ckpt_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tf_tg_model = tf_tg_model.to(device)

criterion = torch.nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, tf_tg_model.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)

profile_training_step_times_bag(
    model=tf_tg_model,
    loader=train_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    n_batches=25,
)

Loading new TFTGBindingModel


/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/lightning_fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
/tmp/ipykernel_2935366/2913531

batch 000 | load=0.0813s | h2d=0.0892s | forward=0.0361s | backward+opt=0.0054s | step=0.2120s | loss=0.6940 | E=8 C=16
batch 001 | load=0.0805s | h2d=0.0155s | forward=0.0361s | backward+opt=0.0040s | step=0.1361s | loss=0.6765 | E=8 C=16
batch 002 | load=0.0802s | h2d=0.0155s | forward=0.0361s | backward+opt=0.0041s | step=0.1359s | loss=0.6721 | E=8 C=16
batch 003 | load=0.0804s | h2d=0.0155s | forward=0.0361s | backward+opt=0.0041s | step=0.1361s | loss=0.6810 | E=8 C=16
batch 004 | load=0.0807s | h2d=0.0155s | forward=0.0361s | backward+opt=0.0041s | step=0.1363s | loss=0.6817 | E=8 C=16
batch 005 | load=0.0803s | h2d=0.0155s | forward=0.0361s | backward+opt=0.0041s | step=0.1360s | loss=0.6782 | E=8 C=16
batch 006 | load=0.0808s | h2d=0.0155s | forward=0.0361s | backward+opt=0.0041s | step=0.1365s | loss=0.6776 | E=8 C=16
batch 007 | load=0.0800s | h2d=0.0155s | forward=0.0362s | backward+opt=0.0041s | step=0.1357s | loss=0.6505 | E=8 C=16
batch 008 | load=0.0807s | h2d=0.0155s |